# Wildlife Strike Prediction — Complete Pipeline (GPU)

Phases 2–5: Data Cleaning → Feature Engineering → Modeling → Submission

**Runtime:** Set to **GPU** (Runtime → Change runtime type → T4 GPU) for XGBoost and PyTorch acceleration.

In [3]:
# --- Setup: install dependencies & locate data files ---
!pip install -q imbalanced-learn xgboost lightgbm catboost pyarrow

import os

try:
    from google.colab import drive, files as colab_files
    IN_COLAB = True
except ModuleNotFoundError:
    drive = None
    colab_files = None
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    # Set your Drive folder path here if your CSVs are in a subfolder.
    DRIVE_FOLDER = '/content/drive/MyDrive/Kaggle'
else:
    # Local fallback: run from the folder containing train.csv and test.csv.
    DRIVE_FOLDER = os.getcwd()

train_path = os.path.join(DRIVE_FOLDER, 'train.csv')
test_path  = os.path.join(DRIVE_FOLDER, 'test.csv')

assert os.path.exists(train_path), f'train.csv not found at {train_path}'
assert os.path.exists(test_path),  f'test.csv not found at {test_path}'
print(f'train.csv: {os.path.getsize(train_path)/1e6:.1f} MB')
print(f'test.csv:  {os.path.getsize(test_path)/1e6:.1f} MB')
print(f'Running in {"Colab" if IN_COLAB else "local"} mode from: {DRIVE_FOLDER}')


train.csv: 134.4 MB
test.csv:  14.9 MB
Running in local mode from: /Users/frankhou/Desktop/ML-course-project/frank_submission(In_Progress)


## Phases 2-3: Data Cleaning & Feature Engineering

This section defines **`feature_engineer(df, null_drop_cols)`**, which applies the **same**
non-target-dependent transforms to train and test so schemas stay aligned and preprocessing
inside this function does not use the test label.

**Traceability:** This notebook follows `AI_GUIDELINES.md`: the project specification, dataset description, and syllabus were checked before these Phase 2-5 changes. The notes below map each feature group to documented columns and keep the modeling choices inside covered course topics.

### Alignment with `AI_GUIDELINES.md`

- **Project spec:** The workflow emphasizes data cleaning, EDA-driven feature engineering, model comparison, hyperparameter tuning, and a runnable final submission. Kaggle predictions are generated directly from model probabilities; there is no manual output editing or external data.
- **Dataset description:** Feature names and meanings are tied to the provided column glossary, including **TIME**, **PHASE_OF_FLIGHT**, **NUM_SEEN**, **NUM_STRUCK**, **SIZE**, aircraft/engine fields, weather fields, report metadata, and **INDICATED_DAMAGE** as the target.
- **Syllabus limits:** Methods stay within supervised learning, feature engineering, cross-validation, regularization, class-imbalance evaluation, boosting/ensembles, and feed-forward neural networks. The PyTorch network is an MLP only; boosting libraries are used for covered boosting concepts.

### End-to-end order

1. **Phase 2** - Mine **REMARKS** / **COMMENTS**; derive compact flags from sparse documentation fields (**RUNWAY**, **LOCATION**, **OUT_OF_RANGE_SPECIES**, **BIRD_BAND_NUMBER**, **FLT**, **TIME**, **ENROUTE_STATE**, **ENG_*_POS**); drop IDs / dates and only the high-null fields that are not protected for modeling; add missingness indicators; impute grouped numeric medians; winsorize **SPEED** / **HEIGHT** / **DISTANCE** where quantiles are valid.
2. **Phase 3** - Temporal encodings; phase risk + unknown-phase flag; wildlife buckets; aircraft / weather / geo; **SOURCE** / **PERSON** / **OPERATOR** role flags; species keywords + **SPECIES_ID** unknown flag; interactions including warning-unknown x night and unknown phase x speed; severity, speed/altitude bands, and multi-engine features.
3. **Next notebook cell** - Group 7 smoothed target encoding (OOF on train) using **`INDICATED_DAMAGE`**. High-cardinality target encodings are kept outside `feature_engineer()` so label-dependent transforms are easy to audit.

### Protected high-null columns

The raw training data has several useful columns with more than 50% missingness. Dropping all of them blindly removes the exact variables used for Phase 3. The cleaning code now protects these domain-relevant fields before applying the high-null drop list:

| Protected column | Why keep it |
|---|---|
| `SPEED`, `HEIGHT`, `NUM_SEEN` | Important strike severity and wildlife exposure signals even when sparse |
| `SKY`, `PRECIPITATION` | Weather and visibility conditions from the data dictionary |
| `LOCATION`, `ENROUTE_STATE`, `BIRD_BAND_NUMBER`, `FLT`, `ENG_3_POS`, `ENG_4_POS` | Converted into compact flags/counts before raw sparse columns are dropped or imputed |

### Feature-group checklist

| # | Theme |
|---|--------|
| 1 | Missingness flags - blanks as signal |
| 2 | Temporal / cyclical - hour, month, season, migration |
| 3 | Phase-of-flight risk - ordinal score + unknown flag |
| 4 | Wildlife - count buckets, size, flock, struck/seen |
| 5 | Aircraft / weather / geo - jet flag, coast proxies, precip, report metadata |
| 6 | Physics-style interactions - kinetic proxies, polynomials |
| 7 | Target encoding - separate OOF cell, including leakage-safe `NUM_ENGS_te` |
| 8 | Impact severity - altitude bins, speed bands, bird kinetic proxy |
| 9 | Multi-engine & triple interactions |


Run the cells below **top to bottom** before loading data.


In [4]:
import pandas as pd
import numpy as np
import warnings
import re
import gc

warnings.filterwarnings("ignore")


### Phase 2: Cleaning

#### Text mining
for REMARKS, & COMMENTS fields: We detect the presence and length of notes. We search for keywords and language that might allude to whether damage is or is not present and composed a _kw_count variable explaining the frequency of those words.


#### Sparse fields
before dropping sparse or string heavy columns, we create compact flags/counts (like runway info present, engine documentation completeness, seasonal date encodings) We then drop the raw fields


#### Dropping
We removed fields that we considered redundant or not applicable as a predictor, as well as fields missing >50% with the exception of a select few we still needed such as SPEED, NUM_SEEN, PRECIPITATION


#### Missingness
We added missingness flags for key fields like SPEED, NUM_ENGINE, geographic fields as well as a total missing fields count becuase missing is reflective of reporting behavior and not damage results. This allows our mnodel to learn even when information was never recorded


#### Imputation
For SPEED & HEIGHT we calculated mediens within AC_MASS. Remaining numerics were replaced with golabl medians. Extreme values in SPEED, HEIGHT, DISTANCE were winsorized to reduce the impact of outliers

In [ ]:
PROTECTED_HIGH_NULL_COLS = {
    # Sparse but directly useful for Phase 3 feature engineering.
    "SPEED",
    "HEIGHT",
    "NUM_SEEN",
    "SKY",
    "PRECIPITATION",
    # Sparse raw fields converted into compact flags before the drop step.
    "LOCATION",
    "ENROUTE_STATE",
    "BIRD_BAND_NUMBER",
    "FLT",
    "ENG_3_POS",
    "ENG_4_POS",
}

RAW_DROP_COLS = [
    "INDEX_NR",
    "REMARKS",
    "COMMENTS",
    "LUPDATE",
    "TRANSFER",
    "BIRD_BAND_NUMBER",
    "AIRPORT",
    "INCIDENT_DATE",
    "FLT",
    "ENROUTE_STATE",
]

CORE_NUMERIC_COLS = [
    "LATITUDE",
    "LONGITUDE",
    "AC_MASS",
    "NUM_ENGS",
    "ENG_1_POS",
    "ENG_2_POS",
    "ENG_3_POS",
    "ENG_4_POS",
    "HEIGHT",
    "SPEED",
    "DISTANCE",
    "TIME_DECIMAL",
    "OUT_OF_RANGE_SPECIES",
    "REMAINS_COLLECTED",
    "REMAINS_SENT",
]


def phase2_mine_remarks_comments(df):
    for text_col in ["REMARKS", "COMMENTS"]:
        if text_col not in df.columns:
            continue
        txt = df[text_col].fillna("").astype(str).str.lower()
        p = text_col[:3].lower()
        df[f"has_{p}"] = (txt.str.len() > 0).astype(np.int8)
        df[f"{p}_len"] = txt.str.len().clip(upper=500)
        df[f"{p}_engine"] = txt.str.contains(
            r"engine|ingest|ingestion|turbine|fan\s*blade|compressor", regex=True
        ).astype(np.int8)
        df[f"{p}_struct"] = txt.str.contains(
            r"dent|crack|hole|shatter|damage|broke|punctur|substantial|deform", regex=True
        ).astype(np.int8)
        df[f"{p}_windshld"] = txt.str.contains(
            r"windshield|canopy|window|radome|nose\s*cone", regex=True
        ).astype(np.int8)
        df[f"{p}_evidence"] = txt.str.contains(
            r"blood|feather|remains|carcass|tissue", regex=True
        ).astype(np.int8)
        df[f"{p}_ops"] = txt.str.contains(
            r"emergency|return|rerout|abort|shutdown|ingestion", regex=True
        ).astype(np.int8)
        df[f"{p}_repair"] = txt.str.contains(
            r"repair|replaced|replacement|removed\s+from\s+service|out\s+of\s+service|aircraft\s+on\s+ground", regex=True
        ).astype(np.int8)
        df[f"{p}_explicit_no_damage"] = txt.str.contains(
            r"no\s+(known\s+|reported\s+|apparent\s+|visible\s+|aircraft\s+)?damage|not\s+damaged|without\s+damage|damage\s*[:\-]?\s*none", regex=True
        ).astype(np.int8)
        df[f"{p}_inspection_only"] = txt.str.contains(
            r"inspect|inspection|checked|returned\s+to\s+service|continued\s+flight|no\s+further", regex=True
        ).astype(np.int8)
        df[f"{p}_kw_count"] = (
            df[f"{p}_engine"]
            + df[f"{p}_struct"]
            + df[f"{p}_windshld"]
            + df[f"{p}_evidence"]
            + df[f"{p}_ops"]
            + df[f"{p}_repair"]
            - df[f"{p}_explicit_no_damage"]
        )
    return df


def phase2_derive_documentation_flags(df):
    """Structured signals from sparse / administrative fields before raw drops."""
    if "OUT_OF_RANGE_SPECIES" in df.columns:
        df["oor_species_flag"] = (
            pd.to_numeric(df["OUT_OF_RANGE_SPECIES"], errors="coerce").fillna(0).astype(np.int8)
        )
        df["MISS_OUT_OF_RANGE_SPECIES"] = df["OUT_OF_RANGE_SPECIES"].isna().astype(np.int8)
        df.drop(columns=["OUT_OF_RANGE_SPECIES"], inplace=True)

    if "BIRD_BAND_NUMBER" in df.columns:
        band = df["BIRD_BAND_NUMBER"].fillna("").astype(str).str.strip()
        df["bird_was_banded"] = (band.str.len() > 0).astype(np.int8)

    if "FLT" in df.columns:
        flt = df["FLT"].fillna("").astype(str).str.strip()
        df["has_flight_number"] = (flt.str.len() > 0).astype(np.int8)

    eng_cols = [c for c in ["ENG_1_POS", "ENG_2_POS", "ENG_3_POS", "ENG_4_POS"] if c in df.columns]
    if eng_cols:
        eng_raw = df[eng_cols].apply(pd.to_numeric, errors="coerce")
        df["eng_pos_nonmissing"] = eng_raw.notna().sum(axis=1).astype(np.int8)
        df["eng_pos_distinct"] = eng_raw.nunique(axis=1).astype(np.int8)

    if "INCIDENT_DATE" in df.columns:
        dt = pd.to_datetime(df["INCIDENT_DATE"], errors="coerce")
        doy = dt.dt.dayofyear.fillna(183).astype(float)
        week = dt.dt.isocalendar().week.astype("float").fillna(26)
        dow = dt.dt.dayofweek.fillna(3).astype(float)
        df["incident_doy_sin"] = np.sin(2 * np.pi * doy / 366.0)
        df["incident_doy_cos"] = np.cos(2 * np.pi * doy / 366.0)
        df["incident_week_sin"] = np.sin(2 * np.pi * week / 53.0)
        df["incident_week_cos"] = np.cos(2 * np.pi * week / 53.0)
        df["incident_is_weekend"] = dow.isin([5.0, 6.0]).astype(np.int8)
        if "LUPDATE" in df.columns:
            upd = pd.to_datetime(df["LUPDATE"], errors="coerce")
            lag = (upd - dt).dt.days
            lag_clean = lag.clip(lower=0).fillna(lag.clip(lower=0).median())
            df["update_lag_days_log1p"] = np.log1p(lag_clean).astype(np.float32)
            df["update_lag_missing"] = lag.isna().astype(np.int8)
            df["update_before_incident"] = (lag < 0).fillna(False).astype(np.int8)
            df["update_lag_very_long"] = (lag_clean > 730).astype(np.int8)
            df["update_lag_extreme"] = (lag_clean > 2400).astype(np.int8)

    if "RUNWAY" in df.columns:
        rw = df["RUNWAY"].fillna("").astype(str).str.strip().str.upper()
        df["has_runway_code"] = (rw.str.len() > 0).astype(np.int8)
        df["runway_len_chars"] = rw.str.len().clip(upper=30).astype(np.float32)
        df["runway_has_number"] = rw.str.contains(r"\d", regex=True).astype(np.int8)
        rnum = pd.to_numeric(rw.str.extract(r"(\d{1,2})", expand=False), errors="coerce")
        heading = (rnum.fillna(18.0).clip(1, 36) * 10.0) % 360.0
        df["runway_heading_sin"] = np.sin(2 * np.pi * heading / 360.0)
        df["runway_heading_cos"] = np.cos(2 * np.pi * heading / 360.0)
        df["runway_side_left"] = rw.str.contains("L", regex=False).astype(np.int8)
        df["runway_side_right"] = rw.str.contains("R", regex=False).astype(np.int8)
        df["runway_side_center"] = rw.str.contains("C", regex=False).astype(np.int8)
        df["runway_water"] = rw.str.contains("W", regex=False).astype(np.int8)
        df.drop(columns=["RUNWAY"], inplace=True)

    if "LOCATION" in df.columns:
        loc = df["LOCATION"].fillna("").astype(str).str.strip()
        loc_l = loc.str.lower()
        df["has_location_text"] = (loc.str.len() > 0).astype(np.int8)
        df["location_text_len"] = loc.str.len().clip(upper=300).astype(np.float32)
        df["loc_mentions_airport"] = loc_l.str.contains(r"airport|runway|rwy", regex=True).astype(np.int8)
        df.drop(columns=["LOCATION"], inplace=True)

    if "TIME" in df.columns:
        t = df["TIME"].fillna("").astype(str).str.strip()
        df["time_string_blank"] = (t.str.len() == 0).astype(np.int8)

    if "ENROUTE_STATE" in df.columns:
        es = df["ENROUTE_STATE"].fillna("").astype(str).str.strip()
        df["has_enroute_state"] = (es.str.len() > 0).astype(np.int8)

    return df


def phase2_drop_and_coerce(df, null_drop_cols):
    protected = set(PROTECTED_HIGH_NULL_COLS)
    high_null_drops = [c for c in null_drop_cols if c not in protected]
    drops = list(set(high_null_drops + RAW_DROP_COLS))
    df.drop(columns=[c for c in drops if c in df.columns], inplace=True)

    for col in CORE_NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def phase2_missingness_flags(df):
    for col in ["SPEED", "HEIGHT", "DISTANCE", "TIME_DECIMAL", "LATITUDE", "LONGITUDE", "AC_MASS", "NUM_ENGS"]:
        if col in df.columns:
            df[f"MISS_{col}"] = df[col].isna().astype("int8")
    mf = [c for c in df.columns if c.startswith("MISS_")]
    if mf:
        df["MISS_total"] = df[mf].sum(axis=1)
    return df


def phase2_impute_and_clip(df):
    for col in CORE_NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "AC_MASS" in df.columns and "SPEED" in df.columns:
        df["SPEED"] = df.groupby("AC_MASS")["SPEED"].transform(lambda x: x.fillna(x.median()))
    if "AC_MASS" in df.columns and "HEIGHT" in df.columns:
        df["HEIGHT"] = df.groupby("AC_MASS")["HEIGHT"].transform(lambda x: x.fillna(x.median()))

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if c != "INDICATED_DAMAGE"]
    for col in num_cols:
        med = df[col].median()
        if pd.isna(med):
            med = 0.0
        df[col] = df[col].fillna(med)

    for col in ["SPEED", "HEIGHT", "DISTANCE"]:
        if col in df.columns:
            lo, hi = df[col].quantile(0.01), df[col].quantile(0.99)
            if pd.notna(lo) and pd.notna(hi) and lo < hi:
                df[col] = df[col].clip(lower=lo, upper=hi)
    return df



### Phase 3 (a): Temporal features, phase risk, wildlife

- **Temporal:** parse **TIME** or use **TIME_DECIMAL**; sine/cosine hour; night / rush /
  dawn–dusk flags; month cyclical features; migration season; long-run year trend.
- **Phase:** map **PHASE_OF_FLIGHT** to an ordinal **phase_risk** score and binary phase
  flags, then drop the raw column.
- **Wildlife:** map **NUM_SEEN** / **NUM_STRUCK** buckets, **SIZE** ordinal, flock and
  ratio features.


In [6]:
def phase3_temporal(df):
    MIG = {8, 9, 10}

    if "TIME" in df.columns and "TIME_DECIMAL" not in df.columns:

        def _parse_hhmm(v):
            if pd.isna(v):
                return np.nan
            s = str(v).strip()
            if not s:
                return np.nan
            m = re.match(r"(\d{1,2}):(\d{2})", s)
            return float(m.group(1)) + float(m.group(2)) / 60.0 if m else np.nan

        df["hour_decimal"] = df["TIME"].apply(_parse_hhmm)
        h = df["hour_decimal"].fillna(12.0)
        df["hour_sin"] = np.sin(2 * np.pi * h / 24)
        df["hour_cos"] = np.cos(2 * np.pi * h / 24)
        df["is_night_strike"] = ((h >= 20) | (h < 6)).astype(np.int8)
        df["is_rush_strike"] = ((h >= 6) & (h <= 9)).astype(np.int8)
        df["is_dawn_dusk"] = (
            (((h >= 5) & (h < 7)) | ((h >= 18) & (h < 20)))
        ).astype(np.int8)
        df.drop(columns=["hour_decimal"], inplace=True)
        df.drop(columns=["TIME"], inplace=True)

    if "TIME_DECIMAL" in df.columns:
        h = df["TIME_DECIMAL"].fillna(12.0)
        df["hour_sin"] = np.sin(2 * np.pi * h / 24)
        df["hour_cos"] = np.cos(2 * np.pi * h / 24)
        df["is_night_strike"] = ((h >= 20) | (h < 6)).astype(np.int8)
        df["is_rush_strike"] = ((h >= 6) & (h <= 9)).astype(np.int8)
        df["is_dawn_dusk"] = (
            (((h >= 5) & (h < 7)) | ((h >= 18) & (h < 20)))
        ).astype(np.int8)
        df.drop(columns=["TIME_DECIMAL"], inplace=True)

    if "TIME" in df.columns:
        df.drop(columns=["TIME"], inplace=True)

    if "INCIDENT_MONTH" in df.columns:
        m = pd.to_numeric(df["INCIDENT_MONTH"], errors="coerce").fillna(6)
        df["month_sin"] = np.sin(2 * np.pi * m / 12)
        df["month_cos"] = np.cos(2 * np.pi * m / 12)
        df["is_migration_season"] = m.isin(MIG).astype(np.int8)
        df["season"] = pd.cut(
            m, bins=[0, 3, 6, 9, 12], labels=[1, 2, 3, 4], include_lowest=True
        ).astype(float)

    if "INCIDENT_YEAR" in df.columns:
        df["year_trend"] = (
            pd.to_numeric(df["INCIDENT_YEAR"], errors="coerce").fillna(2000) - 1990
        ).clip(lower=0)
    if "hour_sin" not in df.columns:
        df["hour_sin"] = 0.0
        df["hour_cos"] = 1.0
    for _c in ["is_night_strike", "is_rush_strike", "is_dawn_dusk"]:
        if _c not in df.columns:
            df[_c] = np.int8(0)
    return df


def phase3_phase_risk(df):
    PR = {
        "Parked": 0.5,
        "Taxi": 1.0,
        "Local": 1.5,
        "Arrival": 2.0,
        "En Route": 2.0,
        "Descent": 2.5,
        "Departure": 2.5,
        "Climb": 3.0,
        "Take-off Run": 4.0,
        "Landing Roll": 4.5,
        "Approach": 5.0,
    }
    if "PHASE_OF_FLIGHT" in df.columns:
        ph = df["PHASE_OF_FLIGHT"].fillna("Unknown").astype(str).str.strip()
        df["phase_is_unknown"] = ph.str.lower().eq("unknown").astype(np.int8)
        df["phase_risk"] = ph.map(PR).fillna(2.5)
        df["is_ground_phase"] = ph.isin(
            {"Taxi", "Parked", "Landing Roll", "Take-off Run"}
        ).astype(np.int8)
        df["is_high_risk_phase"] = ph.isin(
            {"Approach", "Take-off Run", "Landing Roll", "Climb"}
        ).astype(np.int8)
        df.drop(columns=["PHASE_OF_FLIGHT"], inplace=True)
    else:
        df["phase_risk"] = 2.5
        df["is_ground_phase"] = np.int8(0)
        df["is_high_risk_phase"] = np.int8(0)
        df["phase_is_unknown"] = np.int8(0)
    return df


def phase3_wildlife(df):
    RM = {"1": 1.0, "10-Feb": 5.0, "11-100": 30.0, "More than 100": 150.0}

    def _bucket_count(col, new):
        if col not in df.columns:
            df[new] = 1.0
            return
        v = df[col].astype(str).str.strip()
        v = v.replace({"nan": np.nan, "None": np.nan, "<NA>": np.nan, "": np.nan})
        df[new] = v.map(RM).fillna(1.0)
        df.drop(columns=[col], inplace=True)

    _bucket_count("NUM_SEEN", "num_seen")
    _bucket_count("NUM_STRUCK", "num_struck")

    if "SIZE" in df.columns:
        sz = df["SIZE"].fillna("").astype(str).str.strip()
        mp = {"small": 1, "medium": 2, "large": 3}
        df["size_ordinal"] = sz.str.lower().map(mp).fillna(1.0)
        df.drop(columns=["SIZE"], inplace=True)
    else:
        df["size_ordinal"] = 1.0

    df["is_flock"] = (df["num_seen"] > 1).astype(np.int8)
    df["is_large_flock"] = (df["num_seen"] >= 30).astype(np.int8)
    df["struck_seen_ratio"] = (df["num_struck"] / df["num_seen"].replace(0, np.nan)).fillna(
        1.0
    )
    return df


### Phase 3 (b): Aircraft, weather, geography, species keywords

Jet / helicopter / mass / warning flags; **TIME_OF_DAY** and **SKY** scores plus missingness; precipitation
flags; report-source/person/operator indicators; coarse latitude/longitude proxies (coast distances, “northern”, “coastal”);
keyword groups on **SPECIES** (waterfowl, raptor, gull, etc.) crossed with size and jet.


In [7]:
def phase3_aircraft_weather_geo(df):
    if "TYPE_ENG" in df.columns:
        te = df["TYPE_ENG"].fillna("").astype(str).str.strip().str.upper()
        df["is_jet"] = te.isin({"B", "D"}).astype(np.int8)
        df.drop(columns=["TYPE_ENG"], inplace=True)
    if "AC_CLASS" in df.columns:
        ac = df["AC_CLASS"].fillna("").astype(str).str.strip().str.upper()
        df["is_helicopter"] = (ac == "B").astype(np.int8)
        df.drop(columns=["AC_CLASS"], inplace=True)
    if "AC_MASS" in df.columns:
        df["AC_MASS"] = pd.to_numeric(df["AC_MASS"], errors="coerce").fillna(2.0)
        df["is_heavy_aircraft"] = (df["AC_MASS"] >= 4).astype(np.int8)
    if "WARNED" in df.columns:
        w = df["WARNED"].fillna("Unknown").astype(str).str.strip().str.lower()
        df["warned_yes"] = (w == "yes").astype(np.int8)
        df["warned_no"] = (w == "no").astype(np.int8)
        df["warned_unknown"] = (~w.isin({"yes", "no"})).astype(np.int8)
        df.drop(columns=["WARNED"], inplace=True)
    if "TIME_OF_DAY" in df.columns:
        day = df["TIME_OF_DAY"].fillna("").astype(str).str.strip()
        df["tod_missing"] = day.eq("").astype(np.int8)
        df["tod_day"] = day.eq("Day").astype(np.int8)
        df["tod_night"] = day.eq("Night").astype(np.int8)
        df["tod_dawn_dusk"] = day.isin(["Dawn", "Dusk"]).astype(np.int8)
        df["light_level"] = day.map(
            {"Night": 0, "Dawn": 1, "Dusk": 1, "Day": 3}
        ).fillna(1.5)
        df.drop(columns=["TIME_OF_DAY"], inplace=True)
    if "SKY" in df.columns:
        sky = df["SKY"].fillna("").astype(str).str.strip()
        df["sky_missing"] = sky.eq("").astype(np.int8)
        df["sky_score"] = sky.map(
            {"No Cloud": 0, "Some Cloud": 1, "Overcast": 2}
        ).fillna(0.0)
        df.drop(columns=["SKY"], inplace=True)

    PW = {"Rain", "Fog", "Snow", "Hail"}
    if "PRECIPITATION" in df.columns:
        df["precip_missing"] = df["PRECIPITATION"].isna().astype(np.int8)
        df["has_precip"] = df["PRECIPITATION"].apply(
            lambda v: 0 if pd.isna(v) else int(any(k in str(v) for k in PW))
        ).astype(np.int8)
        df["precip_count"] = df["PRECIPITATION"].apply(
            lambda v: 0 if pd.isna(v) else sum(k in str(v) for k in PW)
        ).astype(np.int8)
        df.drop(columns=["PRECIPITATION"], inplace=True)

    for col in ["LATITUDE", "LONGITUDE"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "SOURCE" in df.columns:
        s = df["SOURCE"].fillna("").astype(str).str.lower()
        df["src_faa_form"] = s.str.contains(r"5200|faa form", regex=True).astype(np.int8)
        df["src_air_transport"] = s.str.contains("air transport").astype(np.int8)
        df["src_airport_report"] = s.str.contains("airport report").astype(np.int8)
        df["src_ntsb"] = s.str.contains("ntsb", regex=False).astype(np.int8)
        df["src_asrs"] = s.str.contains("asrs", regex=False).astype(np.int8)
        df["src_engine_mfg"] = s.str.contains("engine manufacturer", regex=False).astype(np.int8)
        df["src_multiple"] = s.str.contains("multiple", regex=False).astype(np.int8)
        df["src_mor"] = s.eq("mor").astype(np.int8)

    if "PERSON" in df.columns:
        p = df["PERSON"].fillna("").astype(str).str.lower()
        df["filer_is_pilot"] = p.str.contains("pilot").astype(np.int8)
        df["filer_is_tower"] = p.str.contains("tower").astype(np.int8)
        df["filer_is_airport_ops"] = p.str.contains(r"airport\s+oper", regex=True).astype(
            np.int8
        )
        df["filer_is_air_transport_ops"] = p.str.contains(r"air\s+transport", regex=True).astype(np.int8)
        df["filer_is_carcass_found"] = p.str.contains("carcass found", regex=False).astype(np.int8)
        df["filer_is_other"] = p.eq("other").astype(np.int8)

    if "OPERATOR" in df.columns:
        op = df["OPERATOR"].fillna("").astype(str).str.strip().str.upper()
        df["operator_unknown"] = op.eq("UNKNOWN").astype(np.int8)

    if "LATITUDE" in df.columns and "LONGITUDE" in df.columns:
        lat = df["LATITUDE"].fillna(37.0)
        lon = df["LONGITUDE"].fillna(-96.0)
        df["dist_east_coast"] = (lon + 75.0).abs()
        df["dist_west_coast"] = (lon + 120.0).abs()
        df["dist_gulf_coast"] = (lat - 29.0).abs()
        df["dist_us_center"] = np.sqrt((lat - 39.5) ** 2 + (lon + 98.35) ** 2)
        df["is_northern"] = (lat >= 42.0).astype(np.int8)
        df["is_coastal"] = ((df["dist_east_coast"] < 3) | (df["dist_west_coast"] < 3)).astype(
            np.int8
        )
    if "is_jet" not in df.columns:
        df["is_jet"] = np.int8(0)
    return df


def phase3_species_taxonomy(df):
    if "SPECIES" in df.columns:
        sp_str = df["SPECIES"].fillna("").astype(str).str.lower()
        df["sp_waterfowl"] = sp_str.str.contains(
            "goose|geese|duck|swan|pelican|crane|heron|egret|cormorant", regex=True
        ).astype(np.int8)
        df["sp_raptor"] = sp_str.str.contains(
            "hawk|eagle|falcon|owl|vulture|osprey|kite|kestrel|harrier", regex=True
        ).astype(np.int8)
        df["sp_gull"] = sp_str.str.contains("gull|tern|skimmer", regex=True).astype(np.int8)
        df["sp_unknown"] = sp_str.str.contains("unknown", regex=True).astype(np.int8)
        df["sp_large_danger"] = (
            (df["sp_waterfowl"] | df["sp_raptor"]) & (df["size_ordinal"] >= 2)
        ).astype(np.int8)
        df["sp_waterfowl_jet"] = (df["sp_waterfowl"] * df["is_jet"]).astype(np.int8)
        df["sp_raptor_jet"] = (df["sp_raptor"] * df["is_jet"]).astype(np.int8)
    if "SPECIES_ID" in df.columns:
        sid = df["SPECIES_ID"].fillna("").astype(str).str.upper()
        df["species_id_is_unk"] = sid.str.startswith("UNK").astype(np.int8)
    return df


def phase3_interactions_polynomials(df):
    sp = pd.to_numeric(df.get("SPEED", pd.Series(0, index=df.index)), errors="coerce").fillna(
        0
    )
    ht = pd.to_numeric(df.get("HEIGHT", pd.Series(0, index=df.index)), errors="coerce").fillna(
        0
    )
    ms = pd.to_numeric(
        df.get("AC_MASS", pd.Series(2, index=df.index)), errors="coerce"
    ).fillna(2)
    df["kinetic_proxy"] = ms * sp**2
    df["speed_x_height"] = sp * ht
    df["strike_mass_index"] = df["size_ordinal"] * df["num_struck"]
    df["risk_composite"] = df["phase_risk"] * df["size_ordinal"] * np.log1p(df["num_struck"])
    df["jet_x_size"] = df["is_jet"] * df["size_ordinal"]
    df["jet_x_flock"] = df["is_jet"] * df["is_flock"]
    if "has_precip" in df.columns and "light_level" in df.columns:
        df["poor_visibility"] = df["has_precip"] * (3 - df["light_level"]).clip(lower=0)
    df["ground_x_flock"] = df["is_ground_phase"] * df["is_flock"]
    if "warned_unknown" in df.columns:
        df["unk_warn_x_night"] = df["warned_unknown"] * df["is_night_strike"]
    if "phase_is_unknown" in df.columns:
        df["unk_phase_x_speed"] = df["phase_is_unknown"] * sp
    for col in ["SPEED", "HEIGHT", "DISTANCE"]:
        if col in df.columns:
            raw = pd.to_numeric(df[col], errors="coerce").clip(lower=0)
            df[f"{col}_log1p"] = np.log1p(raw)
            df[f"{col}_sqrt"] = np.sqrt(raw.fillna(0))
            df[f"{col}_sq"] = raw.fillna(0) ** 2
    df["kinetic_log1p"] = np.log1p(df["kinetic_proxy"].clip(lower=0))
    return df


### Phase 3 (c): Polynomials, severity bands, multi-engine interactions

Kinetic-style composites and log/sqrt polynomial expansions for **SPEED** / **HEIGHT** /
**DISTANCE**; altitude and speed risk bands; **NUM_ENGS** interactions and triple flags. Engine-position availability is derived in Phase 2 before numeric imputation, so it still represents true documentation completeness rather than imputed non-missing values. The **`feature_engineer`** wrapper at the end runs Phase 2 then all Phase 3 blocks in a fixed order.


In [8]:
def phase3_impact_severity(df):
    for col in ["REMAINS_COLLECTED", "REMAINS_SENT"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(np.int8)
    sp = pd.to_numeric(df.get("SPEED", pd.Series(0, index=df.index)), errors="coerce").fillna(0)
    ht = pd.to_numeric(df.get("HEIGHT", pd.Series(0, index=df.index)), errors="coerce").fillna(0)
    df["bird_kinetic"] = df["size_ordinal"] ** 2 * sp
    df["flock_impact"] = df["num_struck"] * df["size_ordinal"] ** 2 * sp
    df["alt_ground"] = (ht == 0).astype(np.int8)
    df["alt_very_low"] = ((ht > 0) & (ht <= 200)).astype(np.int8)
    df["alt_low"] = ((ht > 200) & (ht <= 500)).astype(np.int8)
    df["alt_medium"] = ((ht > 500) & (ht <= 2000)).astype(np.int8)
    df["height_inv"] = 1.0 / (ht + 1.0)
    df["speed_very_high"] = (sp >= 250).astype(np.int8)
    df["speed_high"] = ((sp >= 150) & (sp < 250)).astype(np.int8)
    df["speed_low"] = (sp < 100).astype(np.int8)
    return df


def phase3_multi_engine_triples(df):
    sp = pd.to_numeric(df.get("SPEED", pd.Series(0, index=df.index)), errors="coerce").fillna(0)
    eng_cols = [c for c in ["ENG_1_POS", "ENG_2_POS", "ENG_3_POS", "ENG_4_POS"] if c in df.columns]
    if eng_cols and "eng_pos_nonmissing" not in df.columns:
        # Fallback only; normal path creates this before imputation in Phase 2.
        df["eng_pos_nonmissing"] = df[eng_cols].notna().sum(axis=1).astype(np.int8)

    if "NUM_ENGS" in df.columns:
        df["NUM_ENGS"] = pd.to_numeric(df["NUM_ENGS"], errors="coerce").fillna(1)
        df["is_multi_engine"] = (df["NUM_ENGS"] >= 2).astype(np.int8)
        df["multi_eng_jet"] = df["is_multi_engine"] * df["is_jet"]
        df["engs_x_size"] = df["NUM_ENGS"] * df["size_ordinal"]
    else:
        df["NUM_ENGS"] = 1.0
        df["is_multi_engine"] = np.int8(0)
        df["multi_eng_jet"] = np.int8(0)
        df["engs_x_size"] = df["size_ordinal"]

    df["deadly_combo"] = (
        df["is_jet"]
        * (df["size_ordinal"] >= 2).astype(np.int8)
        * df["is_high_risk_phase"]
    )
    df["size_x_speed"] = df["size_ordinal"] * sp
    df["size_x_phase"] = df["size_ordinal"] * df["phase_risk"]
    df["size_x_alt_inv"] = df["size_ordinal"] * df["height_inv"]
    df["jet_approach_large"] = (
        df["is_jet"]
        * (df["size_ordinal"] == 3).astype(np.int8)
        * (df["phase_risk"] >= 4.0).astype(np.int8)
    )
    df["fast_and_low"] = df["speed_very_high"] * (
        (df["alt_ground"] | df["alt_very_low"]).astype(np.int8)
    )
    return df


def feature_engineer(df, null_drop_cols):
    """Apply Phase 2-3 feature engineering (all non-target-dependent transforms).

    Parameters
    ----------
    df : DataFrame
        Raw train or test data.
    null_drop_cols : list
        Columns with >50% nulls in training data. Protected Phase 3 columns are kept.
    """
    df = df.copy()
    phase2_mine_remarks_comments(df)
    phase2_derive_documentation_flags(df)
    phase2_drop_and_coerce(df, null_drop_cols)
    phase2_missingness_flags(df)
    phase2_impute_and_clip(df)
    phase3_temporal(df)
    phase3_phase_risk(df)
    phase3_wildlife(df)
    phase3_aircraft_weather_geo(df)
    phase3_species_taxonomy(df)
    phase3_interactions_polynomials(df)
    phase3_impact_severity(df)
    phase3_multi_engine_triples(df)
    return df


print("feature_engineer() and Phase 2-3 helpers defined.")


feature_engineer() and Phase 2-3 helpers defined.


### Load data & Group 7 - target encoding

Run this after the cells above so `feature_engineer` is defined. It reads **train.csv** / **test.csv**, applies `feature_engineer` to both frames, then fits **smoothed OOF target encodings** for categorical columns (species, airport, state, ...), plus leakage-safe OOF encoding for **NUM_ENGS**. This step **uses** `INDICATED_DAMAGE` on the training set only through stratified folds.


In [9]:
# --- Load raw data & apply feature engineering ---

def apply_group7_target_encoding(df_train, df_test, smooth=30, random_state=42):
    """Smoothed OOF target encoding + cross-TE features (mutates dataframes)."""
    from sklearn.model_selection import StratifiedKFold

    y_full = df_train["INDICATED_DAMAGE"].astype(int)
    gm = float(y_full.mean())
    te_maps_p3 = {}
    te_counts_p3 = {}
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

    def _as_cat(frame, col, default="Unknown"):
        return frame[col].fillna(default).astype(str).str.strip().replace("", default)

    for df in [df_train, df_test]:
        if "SPECIES" in df.columns:
            sp_str = df["SPECIES"].fillna("Unknown").astype(str).str.lower()
            conditions = [
                sp_str.str.contains("goose|geese|duck|swan|pelican|crane|heron|egret|cormorant", regex=True),
                sp_str.str.contains("hawk|eagle|falcon|owl|vulture|osprey|kite|kestrel", regex=True),
                sp_str.str.contains("gull|tern|skimmer", regex=True),
                sp_str.str.contains("sparrow|finch|swallow|starling|blackbird|robin|wren|warbler", regex=True),
                sp_str.str.contains("pigeon|dove", regex=True),
                sp_str.str.contains("bat", regex=True),
            ]
            choices = ["waterfowl", "raptor", "gull", "songbird", "pigeon", "bat"]
            df["SPECIES_GROUP"] = np.select(conditions, choices, default="other")

        if {"AIRPORT_ID", "SPECIES_GROUP"}.issubset(df.columns):
            df["AIRPORT_SPECIES_GROUP"] = _as_cat(df, "AIRPORT_ID") + "__" + _as_cat(df, "SPECIES_GROUP")
        if {"STATE", "SPECIES_GROUP"}.issubset(df.columns):
            df["STATE_SPECIES_GROUP"] = _as_cat(df, "STATE") + "__" + _as_cat(df, "SPECIES_GROUP")
        if {"OPID", "SPECIES_GROUP"}.issubset(df.columns):
            df["OPID_SPECIES_GROUP"] = _as_cat(df, "OPID") + "__" + _as_cat(df, "SPECIES_GROUP")
        if {"AIRCRAFT", "SPECIES_GROUP"}.issubset(df.columns):
            df["AIRCRAFT_SPECIES_GROUP"] = _as_cat(df, "AIRCRAFT") + "__" + _as_cat(df, "SPECIES_GROUP")
        if {"AMA", "AMO"}.issubset(df.columns):
            df["AIRCRAFT_MAKE_MODEL"] = _as_cat(df, "AMA") + "__" + _as_cat(df, "AMO")
        if {"phase_risk", "SPECIES_GROUP"}.issubset(df.columns):
            phase_bucket = pd.cut(
                df["phase_risk"].astype(float),
                bins=[-np.inf, 1.5, 2.5, 3.5, np.inf],
                labels=["low", "medium", "high", "critical"],
            ).astype(str)
            df["PHASE_SPECIES_GROUP"] = phase_bucket + "__" + _as_cat(df, "SPECIES_GROUP")
            if "SOURCE" in df.columns:
                df["SOURCE_PHASE"] = _as_cat(df, "SOURCE") + "__" + phase_bucket
        if {"SOURCE", "PERSON"}.issubset(df.columns):
            df["SOURCE_PERSON"] = _as_cat(df, "SOURCE") + "__" + _as_cat(df, "PERSON")
        if {"SOURCE", "SPECIES_GROUP"}.issubset(df.columns):
            df["SOURCE_SPECIES_GROUP"] = _as_cat(df, "SOURCE") + "__" + _as_cat(df, "SPECIES_GROUP")

    def _fit_oof_te(col, drop_original=True, add_count=True):
        if col not in df_train.columns:
            return
        train_cats = df_train[col].astype(str).fillna("Unknown")
        test_cats = df_test[col].astype(str).fillna("Unknown") if col in df_test.columns else None
        oof = pd.Series(gm, index=df_train.index, dtype=float)

        for ti, vi in skf.split(df_train, y_full):
            cats = train_cats.iloc[ti]
            ys = y_full.iloc[ti]
            grp = pd.DataFrame({"cat": cats, "y": ys}).groupby("cat")["y"].agg(["sum", "count"])
            grp["enc"] = (grp["sum"] + smooth * gm) / (grp["count"] + smooth)
            oof.iloc[vi] = train_cats.iloc[vi].map(grp["enc"]).fillna(gm).values

        grp_all = pd.DataFrame({"cat": train_cats, "y": y_full}).groupby("cat")["y"].agg(["sum", "count"])
        grp_all["enc"] = (grp_all["sum"] + smooth * gm) / (grp_all["count"] + smooth)
        te_maps_p3[col] = grp_all["enc"].to_dict()
        te_counts_p3[col] = grp_all["count"].to_dict()

        df_train[f"{col}_te"] = oof
        if test_cats is not None:
            df_test[f"{col}_te"] = test_cats.map(te_maps_p3[col]).fillna(gm)
        if add_count:
            train_counts = train_cats.map(te_counts_p3[col]).fillna(0)
            df_train[f"{col}_count_log1p"] = np.log1p(train_counts).astype(np.float32)
            df_train[f"{col}_rare"] = (train_counts <= 3).astype(np.int8)
            if test_cats is not None:
                test_counts = test_cats.map(te_counts_p3[col]).fillna(0)
                df_test[f"{col}_count_log1p"] = np.log1p(test_counts).astype(np.float32)
                df_test[f"{col}_rare"] = (test_counts <= 3).astype(np.int8)
        if drop_original:
            df_train.drop(columns=[col], inplace=True)
            df_test.drop(columns=[col], inplace=True, errors="ignore")
        print(f"  {col:24s} -> {col}_te  (OOF range {oof.min():.3f} - {oof.max():.3f})")

    TE_COLS = [
        "SPECIES", "SPECIES_GROUP", "SPECIES_ID",
        "AIRPORT_ID", "STATE", "OPID", "FAAREGION",
        "SOURCE", "PERSON", "OPERATOR", "SOURCE_PERSON", "SOURCE_PHASE", "SOURCE_SPECIES_GROUP",
        "REG", "AIRCRAFT", "AMA", "AMO", "AIRCRAFT_MAKE_MODEL",
        "AIRPORT_SPECIES_GROUP", "STATE_SPECIES_GROUP", "OPID_SPECIES_GROUP",
        "AIRCRAFT_SPECIES_GROUP", "PHASE_SPECIES_GROUP",
    ]
    KEEP_RAW_FOR_RESIDUAL_ENCODING = {
        "SPECIES", "SPECIES_GROUP", "SPECIES_ID",
        "AIRPORT_ID", "STATE", "OPID", "FAAREGION",
        "SOURCE", "PERSON", "OPERATOR", "SOURCE_PHASE", "SOURCE_SPECIES_GROUP",
        "AIRCRAFT", "AMA", "AMO", "AIRCRAFT_MAKE_MODEL",
        "AIRPORT_SPECIES_GROUP", "STATE_SPECIES_GROUP", "OPID_SPECIES_GROUP",
        "AIRCRAFT_SPECIES_GROUP", "PHASE_SPECIES_GROUP",
    }
    for col in TE_COLS:
        _fit_oof_te(col, drop_original=(col not in KEEP_RAW_FOR_RESIDUAL_ENCODING), add_count=True)

    # NUM_ENGS is numeric and useful raw, so keep it while adding a leakage-safe OOF TE copy.
    _fit_oof_te("NUM_ENGS", drop_original=False, add_count=True)

    for df in [df_train, df_test]:
        if {"SPECIES_te", "STATE_te"}.issubset(df.columns):
            df["species_x_state_risk"] = df["SPECIES_te"] * df["STATE_te"] / gm
        if {"SPECIES_te", "AIRPORT_ID_te"}.issubset(df.columns):
            df["species_x_airport_risk"] = df["SPECIES_te"] * df["AIRPORT_ID_te"] / gm
        if {"SPECIES_te", "phase_risk"}.issubset(df.columns):
            df["species_x_phase"] = df["SPECIES_te"] * df["phase_risk"]
        if {"AIRPORT_ID_te", "season"}.issubset(df.columns):
            df["airport_x_season"] = df["AIRPORT_ID_te"] * df["season"]
        if {"AIRPORT_SPECIES_GROUP_te", "AIRCRAFT_SPECIES_GROUP_te"}.issubset(df.columns):
            df["route_aircraft_species_risk"] = (
                df["AIRPORT_SPECIES_GROUP_te"] * df["AIRCRAFT_SPECIES_GROUP_te"] / gm
            )
        rf = ["is_jet", "is_high_risk_phase", "is_flock", "speed_very_high", "alt_very_low", "alt_ground"]
        present = [f for f in rf if f in df.columns]
        df["risk_flag_count"] = df[present].sum(axis=1).astype(np.int8) if present else np.int8(0)

        entity_te_cols = [
            c for c in [
                "SPECIES_te", "AIRPORT_ID_te", "STATE_te", "OPID_te", "AIRCRAFT_te",
                "AIRPORT_SPECIES_GROUP_te", "AIRCRAFT_SPECIES_GROUP_te", "PHASE_SPECIES_GROUP_te",
            ] if c in df.columns
        ]
        if entity_te_cols:
            df["entity_te_mean"] = df[entity_te_cols].mean(axis=1)
            df["entity_te_max"] = df[entity_te_cols].max(axis=1)
            df["entity_te_min"] = df[entity_te_cols].min(axis=1)
            df["entity_te_range"] = df["entity_te_max"] - df["entity_te_min"]
            df["entity_te_above_global"] = (df[entity_te_cols] > gm).sum(axis=1).astype(np.int8)

    print(f"\nAfter TE - Train: {df_train.shape}, Test: {df_test.shape}")
    return te_maps_p3, gm


train_df = pd.read_csv(train_path, low_memory=False)
test_df  = pd.read_csv(test_path,  low_memory=False)
submission_ids = test_df["INDEX_NR"].copy()
print(f"Raw - Train: {train_df.shape}, Test: {test_df.shape}")

null_drop_cols = train_df.columns[train_df.isnull().mean() > 0.50].tolist()
print("High-null columns protected for FE:", sorted(set(null_drop_cols) & PROTECTED_HIGH_NULL_COLS))

df_train = feature_engineer(train_df, null_drop_cols)
df_test  = feature_engineer(test_df,  null_drop_cols)
del train_df, test_df; gc.collect()
print(f"After FE - Train: {df_train.shape}, Test: {df_test.shape}")

# --- Group 7: Smoothed Target Encoding (OOF for train, full-map for test) ---

te_maps_p3, gm = apply_group7_target_encoding(df_train, df_test, smooth=30, random_state=42)

# Final feature alignment before Phase 4.
feature_cols = [c for c in df_train.columns if c != "INDICATED_DAMAGE"]
missing_in_test = [c for c in feature_cols if c not in df_test.columns]
extra_in_test = [c for c in df_test.columns if c not in feature_cols]
for c in missing_in_test:
    df_test[c] = 0
if extra_in_test:
    df_test.drop(columns=extra_in_test, inplace=True)
df_test = df_test[feature_cols]
print(f"Aligned features - Train X: {(df_train.shape[0], len(feature_cols))}, Test: {df_test.shape}")


Raw - Train: (307178, 55), Test: (34131, 54)
High-null columns protected for FE: ['BIRD_BAND_NUMBER', 'ENG_3_POS', 'ENG_4_POS', 'ENROUTE_STATE', 'FLT', 'HEIGHT', 'LOCATION', 'NUM_SEEN', 'PRECIPITATION', 'SKY', 'SPEED']
After FE - Train: (307178, 195), Test: (34131, 194)
  SPECIES                  -> SPECIES_te  (OOF range 0.002 - 0.798)
  SPECIES_GROUP            -> SPECIES_GROUP_te  (OOF range 0.010 - 0.307)
  SPECIES_ID               -> SPECIES_ID_te  (OOF range 0.002 - 0.798)
  AIRPORT_ID               -> AIRPORT_ID_te  (OOF range 0.008 - 0.389)
  STATE                    -> STATE_te  (OOF range 0.010 - 0.156)
  OPID                     -> OPID_te  (OOF range 0.002 - 0.370)
  FAAREGION                -> FAAREGION_te  (OOF range 0.046 - 0.126)
  SOURCE                   -> SOURCE_te  (OOF range 0.010 - 0.791)
  PERSON                   -> PERSON_te  (OOF range 0.000 - 0.190)
  OPERATOR                 -> OPERATOR_te  (OOF range 0.002 - 0.370)
  SOURCE_PERSON            -> SOURCE_PERS

## Phase 4: Modeling & Hyperparameter Tuning

We split the training data 80/20 for initial model development, encode remaining categorical
columns (OOF target encoding + frequency + ordinal), and scale for linear/MLP models.

**Individual model cells** (below) train each model once for quick iteration and narrative:
1. **Logistic Regression** — linear baseline
2. **XGBoost** (GPU) — second-order gradient boosting
3. **PyTorch MLP** (GPU) — feed-forward network with ReLU, dropout, and L2 weight decay
4. **HistGradientBoosting** — sklearn histogram-based ensemble
5. **LightGBM** (GPU) — leaf-wise boosting
6. **CatBoost** (GPU, CPU fallback) — ordered boosting with symmetric trees
7. **XGBoost DART** (GPU) — dropout-regularized boosted trees for diversity
8. **XGBoost (tuned)** — RandomizedSearchCV over hyperparameter space

**Final ensemble** uses budgeted K-fold OOF stacking with a LogisticRegression meta-learner
(see stacking cell below), then retrains all models on 100% data for submission. Runtime constants in the first Phase 4 code cell cap repeated seeds, folds, boosting rounds, and MLP epochs so training remains meaningful without spending hours on tiny marginal gains.


In [10]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, balanced_accuracy_score


# High-accuracy run controls. This version intentionally spends more time on
# the strongest GPU-friendly tree models and seed averaging.
# Keep 5 folds here to match the OOF target-encoding folds above; this avoids
# training-fold labels leaking into stacking validation encodings.
STACK_FOLDS = 5
TREE_SEEDS = [42, 2024, 777]
CATBOOST_SEEDS = [42, 2024, 777]
EARLY_STOP_ROUNDS = 125
TRAIN_LOG_EVERY = 250

XGB_ESTIMATORS = 3200
XGB_LR = 0.008
LGB_ESTIMATORS = 3400
LGB_LR = 0.008
LGB_ALT_ESTIMATORS = 4200
LGB_ALT_LR = 0.006
CAT_ITERATIONS = 2600
CAT_RAW_ITERATIONS = 2200
CATBOOST_RAW_SEEDS = [42, 2024]
DART_ESTIMATORS = 1400
EXTRA_TREES_N = 320

MLP_MAX_EPOCHS = 100
MLP_PATIENCE = 12
STACK_MLP_MAX_EPOCHS = 45
STACK_MLP_PATIENCE = 10
FINAL_MLP_EPOCHS = 45

XGB_SEARCH_ITER = 24
XGB_SEARCH_SUBSAMPLE = 75_000


def encode_remaining_categoricals(X_train_raw, y_train, X_valid_raw=None, X_test_raw=None, *, smooth=5, random_state=42):
    """Encode residual object columns with OOF TE, frequency, and ordinal features."""
    X_train = X_train_raw.copy()
    X_valid = None if X_valid_raw is None else X_valid_raw.copy()
    X_test = None if X_test_raw is None else X_test_raw.copy()
    frames = [f for f in [X_train, X_valid, X_test] if f is not None]
    obj_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    print(f'Categorical columns to encode ({len(obj_cols)}): {obj_cols}')

    if obj_cols:
        global_mean = float(y_train.mean())
        skf_enc = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

        for col in obj_cols:
            for frame in frames:
                frame[col] = frame[col].fillna('__NA__').astype(str)

            X_train[f'{col}_te'] = global_mean
            for ti, vi in skf_enc.split(X_train, y_train):
                cats = X_train[col].iloc[ti]
                ys = y_train.iloc[ti]
                agg = pd.DataFrame({'cat': cats, 'y': ys}).groupby('cat')['y'].agg(['sum', 'count'])
                enc = (agg['sum'] + smooth * global_mean) / (agg['count'] + smooth)
                X_train.iloc[vi, X_train.columns.get_loc(f'{col}_te')] = (
                    X_train[col].iloc[vi].map(enc).fillna(global_mean).values
                )

            full_agg = pd.DataFrame({'cat': X_train[col], 'y': y_train}).groupby('cat')['y'].agg(['sum', 'count'])
            full_enc = (full_agg['sum'] + smooth * global_mean) / (full_agg['count'] + smooth)
            full_map = full_enc.to_dict()
            count_map = full_agg['count'].to_dict()

            for frame in [X_valid, X_test]:
                if frame is not None:
                    frame[f'{col}_te'] = frame[col].map(full_map).fillna(global_mean)

            for frame in frames:
                frame[f'{col}_freq'] = frame[col].map(count_map).fillna(0).astype(np.float32)
                frame[f'{col}_freq_log1p'] = np.log1p(frame[f'{col}_freq']).astype(np.float32)

        oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        oe_train = pd.DataFrame(
            oe.fit_transform(X_train[obj_cols]),
            columns=[f'{c}_oe' for c in obj_cols],
            index=X_train.index,
        )
        X_train = pd.concat([X_train.drop(columns=obj_cols), oe_train], axis=1)

        if X_valid is not None:
            oe_valid = pd.DataFrame(
                oe.transform(X_valid[obj_cols]),
                columns=[f'{c}_oe' for c in obj_cols],
                index=X_valid.index,
            )
            X_valid = pd.concat([X_valid.drop(columns=obj_cols), oe_valid], axis=1)

        if X_test is not None:
            oe_test = pd.DataFrame(
                oe.transform(X_test[obj_cols]),
                columns=[f'{c}_oe' for c in obj_cols],
                index=X_test.index,
            )
            X_test = pd.concat([X_test.drop(columns=obj_cols), oe_test], axis=1)

    for frame in [f for f in [X_train, X_valid, X_test] if f is not None]:
        for c in frame.columns:
            frame[c] = pd.to_numeric(frame[c], errors='coerce')

    for frame in [X_valid, X_test]:
        if frame is None:
            continue
        for c in X_train.columns:
            if c not in frame.columns:
                frame[c] = 0
        extra = [c for c in frame.columns if c not in X_train.columns]
        if extra:
            frame.drop(columns=extra, inplace=True)
        frame = frame[X_train.columns]
        if X_valid is not None and frame.index.equals(X_valid.index):
            X_valid = frame
        elif X_test is not None and frame.index.equals(X_test.index):
            X_test = frame

    return X_train, X_valid, X_test, obj_cols


y_all = df_train['INDICATED_DAMAGE']
X_base = df_train.drop(columns=['INDICATED_DAMAGE'])
X_sub_base = df_test.copy()
del df_train, df_test; gc.collect()

X_trn_base, X_val_base, y_trn, y_val = train_test_split(
    X_base, y_all, test_size=0.20, random_state=42, stratify=y_all
)

# Holdout matrices for the individual model cells.
X_trn, X_val, _, obj_cols = encode_remaining_categoricals(
    X_trn_base, y_trn, X_val_base, None, smooth=5, random_state=42
)

# Full-data OOF matrix for stacking and final retraining. The competition test map now uses 100% of training labels.
X_all, _, X_sub, full_obj_cols = encode_remaining_categoricals(
    X_base, y_all, None, X_sub_base, smooth=5, random_state=42
)

print(f'Feature matrix — train: {X_trn.shape}, val: {X_val.shape}, full: {X_all.shape}, test: {X_sub.shape}')
print(f'Target balance (train): {y_trn.value_counts(normalize=True).to_string()}')
print(f'Full/test columns aligned: {list(X_all.columns) == list(X_sub.columns)}')

# Scaled versions for LR / MLP (NaN filled for linear models)
scaler = StandardScaler()
X_trn_sc = pd.DataFrame(scaler.fit_transform(X_trn.fillna(0)),
                         columns=X_trn.columns, index=X_trn.index)
X_val_sc = pd.DataFrame(scaler.transform(X_val.fillna(0)),
                         columns=X_val.columns, index=X_val.index)
print('StandardScaler applied (separate scaled copies for LR/MLP)')


Categorical columns to encode (21): ['AIRPORT_ID', 'STATE', 'FAAREGION', 'OPID', 'OPERATOR', 'AIRCRAFT', 'AMA', 'AMO', 'SPECIES_ID', 'SPECIES', 'SOURCE', 'PERSON', 'SPECIES_GROUP', 'AIRPORT_SPECIES_GROUP', 'STATE_SPECIES_GROUP', 'OPID_SPECIES_GROUP', 'AIRCRAFT_SPECIES_GROUP', 'AIRCRAFT_MAKE_MODEL', 'PHASE_SPECIES_GROUP', 'SOURCE_PHASE', 'SOURCE_SPECIES_GROUP']
Categorical columns to encode (21): ['AIRPORT_ID', 'STATE', 'FAAREGION', 'OPID', 'OPERATOR', 'AIRCRAFT', 'AMA', 'AMO', 'SPECIES_ID', 'SPECIES', 'SOURCE', 'PERSON', 'SPECIES_GROUP', 'AIRPORT_SPECIES_GROUP', 'STATE_SPECIES_GROUP', 'OPID_SPECIES_GROUP', 'AIRCRAFT_SPECIES_GROUP', 'AIRCRAFT_MAKE_MODEL', 'PHASE_SPECIES_GROUP', 'SOURCE_PHASE', 'SOURCE_SPECIES_GROUP']
Feature matrix — train: (245742, 327), val: (61436, 327), full: (307178, 327), test: (34131, 327)
Target balance (train): INDICATED_DAMAGE
0    0.936429
1    0.063571
Full/test columns aligned: True
StandardScaler applied (separate scaled copies for LR/MLP)


In [11]:
# --- Evaluation helpers ---

def find_best_threshold(y_true, probs, lo=0.01, hi=0.99, step=0.005):
    y = np.asarray(y_true).astype(int)
    p = np.nan_to_num(np.asarray(probs, dtype=float), nan=-np.inf)
    thresholds = np.arange(lo, hi + step / 2, step)

    total_pos = int(y.sum())
    total_neg = int(len(y) - total_pos)
    if len(y) == 0 or total_pos == 0 or total_neg == 0 or thresholds.size == 0:
        fallback = 0.5
        return fallback, balanced_accuracy_score(y, (p >= fallback).astype(int))

    order = np.argsort(-p)
    p_desc = p[order]
    y_desc = y[order]
    tp_cum = np.cumsum(y_desc)
    fp_cum = np.cumsum(1 - y_desc)

    # Number of rows predicted positive for each threshold.
    k = np.searchsorted(-p_desc, -thresholds, side='right')
    tp = np.where(k > 0, tp_cum[k - 1], 0)
    fp = np.where(k > 0, fp_cum[k - 1], 0)
    tn = total_neg - fp

    ba = 0.5 * ((tp / total_pos) + (tn / total_neg))
    best_idx = int(np.nanargmax(ba))
    return float(thresholds[best_idx]), float(ba[best_idx])

def eval_model(name, y_true, probs):
    t, ba = find_best_threshold(y_true, probs)
    preds = (probs >= t).astype(int)
    auc = roc_auc_score(y_true, probs)
    print(f'\n[{name}] Best threshold={t:.3f}')
    print(f'Balanced Accuracy : {ba:.4f}')
    print(f'ROC-AUC           : {auc:.4f}')
    print(classification_report(y_true, preds))
    return t, ba, auc, probs


### Model 1: Baseline — Logistic Regression
A simple linear baseline to establish the performance floor. `class_weight='balanced'` internally adjusts to our ~6% minority class.

In [12]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(solver='saga', max_iter=300, random_state=42,
                        class_weight='balanced', n_jobs=-1)
lr.fit(X_trn_sc, y_trn)

lr_probs = lr.predict_proba(X_val_sc)[:, 1]
lr_t, lr_ba, lr_auc, _ = eval_model('Logistic Regression', y_val, lr_probs)



[Logistic Regression] Best threshold=0.495
Balanced Accuracy : 0.8563
ROC-AUC           : 0.9322
              precision    recall  f1-score   support

           0       0.99      0.87      0.93     57531
           1       0.31      0.84      0.45      3905

    accuracy                           0.87     61436
   macro avg       0.65      0.86      0.69     61436
weighted avg       0.94      0.87      0.90     61436



### Model 2: XGBoost — GPU-Accelerated Gradient Boosting
XGBoost uses second-order Taylor expansion of the loss function and native NaN handling.
The `device='cuda'` parameter offloads tree construction to the GPU.
Early stopping on a held-out validation set prevents overfitting.

In [13]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=XGB_ESTIMATORS,
    max_depth=8,
    learning_rate=XGB_LR,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_weight=10,
    scale_pos_weight=(y_trn == 0).sum() / (y_trn == 1).sum(),
    max_delta_step=1,
    random_state=42,
    n_jobs=-1,
    eval_metric='auc',
    tree_method='hist',
    device='cuda',
    early_stopping_rounds=EARLY_STOP_ROUNDS,
)

# Train on FULL X_trn, monitor early stopping on X_val
xgb_model.fit(X_trn, y_trn,
              eval_set=[(X_val, y_val)],
              verbose=TRAIN_LOG_EVERY)
print(f'XGBoost stopped at iteration {xgb_model.best_iteration}')

xgb_probs = xgb_model.predict_proba(X_val)[:, 1]
xgb_t, xgb_ba, xgb_auc, _ = eval_model('XGBoost', y_val, xgb_probs)



[0]	validation_0-auc:0.87779
[250]	validation_0-auc:0.93507
[500]	validation_0-auc:0.93939
[750]	validation_0-auc:0.94190
[1000]	validation_0-auc:0.94318
[1250]	validation_0-auc:0.94381
[1500]	validation_0-auc:0.94408
[1750]	validation_0-auc:0.94429
[1867]	validation_0-auc:0.94426
XGBoost stopped at iteration 1742

[XGBoost] Best threshold=0.385
Balanced Accuracy : 0.8704
ROC-AUC           : 0.9443
              precision    recall  f1-score   support

           0       0.99      0.88      0.93     57531
           1       0.32      0.86      0.47      3905

    accuracy                           0.88     61436
   macro avg       0.66      0.87      0.70     61436
weighted avg       0.95      0.88      0.90     61436



### Model 3: PyTorch Feed-Forward MLP (GPU)
Feed-forward MLP (input → 512 → 256 → 128 → 64 → 1) with course-safe neural-network components:
- **Dropout** between hidden layers
- **ReLU** hidden activations
- **L2 weight decay** via AdamW
- **Class-weighted BCE loss** — one minority sample worth ~15× a majority sample
- **Early stopping** on validation ROC-AUC

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

class WildlifeMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 512),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x)

n_features = X_trn_sc.shape[1]
mlp = WildlifeMLP(n_features).to(device)

neg_count = (y_trn == 0).sum()
pos_count = (y_trn == 1).sum()
pos_weight = torch.tensor([neg_count / pos_count], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(mlp.parameters(), lr=5e-4, weight_decay=5e-4)
X_tr_t = torch.tensor(X_trn_sc.values, dtype=torch.float32)
y_tr_t = torch.tensor(y_trn.values, dtype=torch.float32).unsqueeze(1)
X_va_t = torch.tensor(X_val_sc.values, dtype=torch.float32)

train_ds = TensorDataset(X_tr_t, y_tr_t)
train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)

best_auc = 0
patience, wait = MLP_PATIENCE, 0

for epoch in range(MLP_MAX_EPOCHS):
    mlp.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward()
        optimizer.step()
    mlp.eval()
    with torch.no_grad():
        val_logits = mlp(X_va_t.to(device)).cpu()
        val_probs = torch.sigmoid(val_logits).numpy().ravel()
        auc = roc_auc_score(y_val, val_probs)
    if auc > best_auc:
        best_auc = auc
        best_state = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}
        wait = 0
    else:
        wait += 1
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d} | AUC={auc:.4f} | best={best_auc:.4f}')
    if wait >= patience:
        print(f'Early stopping at epoch {epoch+1}')
        break

mlp.load_state_dict(best_state)
mlp.eval()
with torch.no_grad():
    mlp_probs = torch.sigmoid(mlp(X_va_t.to(device)).cpu()).numpy().ravel()

mlp_t, mlp_ba, mlp_auc, _ = eval_model('PyTorch MLP', y_val, mlp_probs)


Using device: cpu


### Model 4: HistGradientBoosting (sklearn)
Complementary tree ensemble using histogram-based splits. Supports native NaN handling
and built-in early stopping.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier

hgb = HistGradientBoostingClassifier(
    learning_rate=0.04, max_iter=1200, max_depth=10,
    max_leaf_nodes=255, min_samples_leaf=15,
    l2_regularization=0.05, class_weight='balanced',
    early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=50, random_state=42
)
hgb.fit(X_trn, y_trn)
print(f'HGB stopped at {hgb.n_iter_} iterations')

hgb_probs = hgb.predict_proba(X_val)[:, 1]
hgb_t, hgb_ba, hgb_auc, _ = eval_model('HistGradientBoosting', y_val, hgb_probs)



### Model 5: LightGBM (GPU)
LightGBM uses leaf-wise tree growth (vs XGBoost's level-wise). Here it receives the same encoded feature matrix as the other tree models, giving the ensemble another boosting implementation to compare against without introducing an out-of-syllabus method.

In [ ]:
import lightgbm as lgb


def fit_lgbm_classifier(
    X_train,
    y_train,
    X_eval=None,
    y_eval=None,
    seed=42,
    n_estimators=LGB_ESTIMATORS,
    learning_rate=LGB_LR,
    early_stopping_rounds=EARLY_STOP_ROUNDS,
    log_eval=TRAIN_LOG_EVERY,
    param_overrides=None,
):
    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    base_params = dict(
        n_estimators=n_estimators,
        max_depth=-1,
        num_leaves=255,
        learning_rate=learning_rate,
        subsample=0.85,
        colsample_bytree=0.8,
        min_child_samples=15,
        scale_pos_weight=neg / pos,
        reg_alpha=0.05,
        reg_lambda=0.8,
        random_state=seed,
        n_jobs=-1,
        verbose=-1,
    )
    if param_overrides:
        base_params.update(param_overrides)
    eval_set = [(X_eval, y_eval)] if X_eval is not None and y_eval is not None else None
    callbacks = []
    if eval_set is not None and early_stopping_rounds:
        callbacks.append(lgb.early_stopping(early_stopping_rounds, verbose=bool(log_eval)))
    if eval_set is not None and log_eval:
        callbacks.append(lgb.log_evaluation(log_eval))

    for device in ["gpu", "cpu"]:
        try:
            model = lgb.LGBMClassifier(**base_params, device=device)
            model.fit(X_train, y_train, eval_set=eval_set, callbacks=callbacks or None)
            print(f"LightGBM trained with device={device}")
            return model
        except Exception as e:
            if device == "gpu":
                print(f"LightGBM GPU unavailable ({type(e).__name__}: {e}); retrying on CPU...")
            else:
                raise


lgb_model = fit_lgbm_classifier(
    X_trn,
    y_trn,
    X_val,
    y_val,
    seed=42,
    n_estimators=LGB_ESTIMATORS,
    learning_rate=LGB_LR,
    early_stopping_rounds=EARLY_STOP_ROUNDS,
    log_eval=TRAIN_LOG_EVERY,
)
print(f'LightGBM stopped at iteration {lgb_model.best_iteration_}')

lgb_probs = lgb_model.predict_proba(X_val)[:, 1]
lgb_t, lgb_ba, lgb_auc, _ = eval_model('LightGBM', y_val, lgb_probs)


### Model 6: CatBoost (GPU with CPU fallback)
CatBoost brings an ordered-boosting tree learner that is often strong on tabular problems and tends to make errors different from XGBoost/LightGBM. The helper below tries GPU first for Colab T4, then falls back to CPU if the Colab CatBoost build cannot initialize CUDA.


In [ ]:
from catboost import CatBoostClassifier, Pool


def fit_catboost(X_train, y_train, X_eval=None, y_eval=None, seed=42, iterations=CAT_ITERATIONS):
    params = dict(
        iterations=iterations,
        depth=8,
        learning_rate=0.02,
        l2_leaf_reg=6.0,
        random_strength=1.0,
        bagging_temperature=0.5,
        border_count=128,
        loss_function='Logloss',
        eval_metric='AUC',
        auto_class_weights='Balanced',
        random_seed=seed,
        allow_writing_files=False,
        verbose=TRAIN_LOG_EVERY,
    )
    eval_set = (X_eval.fillna(0), y_eval) if X_eval is not None and y_eval is not None else None
    train_pool = X_train.fillna(0)

    for task_type in ['GPU', 'CPU']:
        try:
            model = CatBoostClassifier(**params, task_type=task_type)
            if eval_set is not None:
                model.fit(train_pool, y_train, eval_set=eval_set, early_stopping_rounds=EARLY_STOP_ROUNDS, use_best_model=True)
            else:
                model.fit(train_pool, y_train)
            print(f'CatBoost trained with task_type={task_type}')
            return model
        except Exception as e:
            if task_type == 'GPU':
                print(f'CatBoost GPU unavailable ({type(e).__name__}: {e}); retrying on CPU...')
            else:
                raise



def prepare_catboost_frame(X, cat_cols=None):
    X_cb = X.copy()
    if cat_cols is None:
        cat_cols = X_cb.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    cat_cols = [c for c in cat_cols if c in X_cb.columns]
    for col in X_cb.columns:
        if col in cat_cols:
            X_cb[col] = X_cb[col].fillna("__NA__").astype(str)
        else:
            X_cb[col] = pd.to_numeric(X_cb[col], errors="coerce").fillna(0)
    return X_cb, cat_cols


def fit_catboost_raw(X_train, y_train, X_eval=None, y_eval=None, seed=42, iterations=CAT_RAW_ITERATIONS):
    X_train_cb, cat_cols = prepare_catboost_frame(X_train)
    train_pool = Pool(X_train_cb, y_train, cat_features=cat_cols)
    eval_pool = None
    if X_eval is not None and y_eval is not None:
        X_eval_cb, _ = prepare_catboost_frame(X_eval, cat_cols=cat_cols)
        eval_pool = Pool(X_eval_cb, y_eval, cat_features=cat_cols)

    params = dict(
        iterations=iterations,
        depth=8,
        learning_rate=0.018,
        l2_leaf_reg=7.0,
        random_strength=1.1,
        bagging_temperature=0.6,
        border_count=128,
        loss_function="Logloss",
        eval_metric="AUC",
        auto_class_weights="Balanced",
        random_seed=seed,
        allow_writing_files=False,
        verbose=TRAIN_LOG_EVERY,
    )

    for task_type in ["GPU", "CPU"]:
        try:
            model = CatBoostClassifier(**params, task_type=task_type)
            if eval_pool is not None:
                model.fit(train_pool, eval_set=eval_pool, early_stopping_rounds=EARLY_STOP_ROUNDS, use_best_model=True)
            else:
                model.fit(train_pool)
            model._raw_cat_cols = cat_cols
            print(f"CatBoost raw trained with task_type={task_type}, cat_cols={len(cat_cols)}")
            return model
        except Exception as e:
            if task_type == "GPU":
                print(f"CatBoost raw GPU unavailable ({type(e).__name__}: {e}); retrying on CPU...")
            else:
                raise


def predict_catboost_raw(model, X):
    X_cb, cat_cols = prepare_catboost_frame(X, cat_cols=getattr(model, "_raw_cat_cols", None))
    pool = Pool(X_cb, cat_features=cat_cols)
    return model.predict_proba(pool)[:, 1]

cat_model = fit_catboost(X_trn, y_trn, X_val, y_val, seed=42, iterations=CAT_ITERATIONS)
cat_probs = cat_model.predict_proba(X_val.fillna(0))[:, 1]
cat_t, cat_ba, cat_auc, _ = eval_model('CatBoost', y_val, cat_probs)


### Model 7: XGBoost DART (GPU)
DART randomly drops boosted trees during training, which regularizes the model and usually creates predictions that are less correlated with standard gradient boosting. That makes it useful as a stacking member even when its standalone score is similar.


In [ ]:
xgb_dart = xgb.XGBClassifier(
    n_estimators=DART_ESTIMATORS,
    max_depth=7,
    learning_rate=0.012,
    subsample=0.85,
    colsample_bytree=0.75,
    min_child_weight=8,
    scale_pos_weight=(y_trn == 0).sum() / (y_trn == 1).sum(),
    max_delta_step=1,
    reg_alpha=0.05,
    reg_lambda=1.5,
    gamma=0.05,
    booster='dart',
    sample_type='uniform',
    normalize_type='tree',
    rate_drop=0.08,
    skip_drop=0.55,
    random_state=42,
    n_jobs=-1,
    eval_metric='auc',
    tree_method='hist',
    device='cuda',
    early_stopping_rounds=EARLY_STOP_ROUNDS,
)

xgb_dart.fit(X_trn, y_trn, eval_set=[(X_val, y_val)], verbose=TRAIN_LOG_EVERY)
print(f'XGBoost DART stopped at iteration {xgb_dart.best_iteration}')

xgb_dart_probs = xgb_dart.predict_proba(X_val)[:, 1]
xgb_dart_t, xgb_dart_ba, xgb_dart_auc, _ = eval_model('XGBoost DART', y_val, xgb_dart_probs)



### XGBoost Hyperparameter Tuning (GPU)
RandomizedSearchCV over a wide parameter space targeting threshold-tuned balanced accuracy.
Uses a capped subsample and limited random trials for a useful search without letting tuning dominate total runtime.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
 

param_grid = {
    'n_estimators': [1400, 2200, 3200, 4200],
    'max_depth': [5, 6, 7, 8, 10],
    'learning_rate': [0.004, 0.006, 0.008, 0.01, 0.015, 0.02],
    'scale_pos_weight': [6, 8, 10, 12, 14, 16, 18, 20],
    'max_delta_step': [0, 1, 3, 5, 7],
    'subsample': [0.65, 0.75, 0.85, 0.95],
    'colsample_bytree': [0.55, 0.65, 0.75, 0.85],
    'min_child_weight': [3, 5, 8, 10, 15, 20],
    'gamma': [0, 0.05, 0.1, 0.2, 0.4],
    'reg_alpha': [0, 0.05, 0.1, 0.3, 0.6],
    'reg_lambda': [0.5, 0.8, 1.0, 1.5, 2.5],
}

rng = np.random.RandomState(42)
n_sub = min(XGB_SEARCH_SUBSAMPLE, len(X_trn))
idx = rng.choice(len(X_trn), size=n_sub, replace=False)
X_rs = X_trn.fillna(0).iloc[idx]
y_rs = y_trn.iloc[idx]

def threshold_balanced_accuracy_scorer(estimator, X, y):
    probs = estimator.predict_proba(X)[:, 1]
    return find_best_threshold(y, probs, lo=0.01, hi=0.99, step=0.005)[1]


print(f'Running RandomizedSearchCV ({XGB_SEARCH_ITER} iter x 3-fold on {n_sub:,} samples) ...')
rs = RandomizedSearchCV(
    xgb.XGBClassifier(random_state=42, n_jobs=-1, eval_metric='auc',
                       tree_method='hist', device='cuda'),
    param_grid,
    n_iter=XGB_SEARCH_ITER, cv=3, scoring=threshold_balanced_accuracy_scorer,
    random_state=42, n_jobs=1, verbose=1
)
rs.fit(X_rs, y_rs)

print(f'\nBest CV threshold-tuned balanced accuracy: {rs.best_score_:.4f}')
print(f'Best params: {rs.best_params_}')

xgb_tuned = xgb.XGBClassifier(**rs.best_params_, random_state=42, n_jobs=-1,
                                eval_metric='auc', tree_method='hist', device='cuda')
xgb_tuned.fit(X_trn, y_trn)

xgbt_probs = xgb_tuned.predict_proba(X_val)[:, 1]
xgbt_t, xgbt_ba, xgbt_auc, _ = eval_model('XGBoost (tuned)', y_val, xgbt_probs)



### K-Fold OOF Stacking Ensemble

Instead of a simple weighted blend, we use **stacking** — a two-level ensemble:

1. **Level 0 (base models):** budgeted K-fold CV trains all stacking models on each fold, collecting out-of-fold (OOF) predictions for all training rows. XGBoost and LightGBM use two seeds by default; slower CatBoost and DART runs are capped to avoid low-return runtime.
2. **Level 1 (meta-learner):** A LogisticRegression is trained on the OOF probability columns to learn optimal blending — including interactions between models.
3. **Threshold:** Optimized on all OOF predictions (much more stable than a single 20% split).
4. **Submission:** All base models are retrained on 100% of training data, then predictions are fed through the meta-learner.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  K-Fold OOF Stacking Ensemble
#  - Budgeted K-fold CV generates OOF predictions from all base models
#  - Multi-seed averaging for tree models reduces variance
#  - LogisticRegression meta-learner on stacked OOF probabilities
#  - Threshold optimized on ALL OOF predictions (stable)
# ═══════════════════════════════════════════════════════════════════
import time

N = len(y_all)
n_folds = STACK_FOLDS
SEEDS = TREE_SEEDS
CAT_SEEDS = CATBOOST_SEEDS

stack_model_names = ['LR', 'XGB', 'HGB', 'LGB', 'MLP', 'XGB_tuned', 'CatBoost', 'XGB_DART', 'ExtraTrees', 'LGB_alt', 'CatBoost_raw']
oof_preds = np.zeros((N, len(stack_model_names)), dtype=np.float64)

skf_stack = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

t0 = time.time()
for fold_i, (trn_idx, val_idx) in enumerate(skf_stack.split(X_all, y_all)):
    print(f'\n{"="*50} FOLD {fold_i+1}/{n_folds} {"="*50}')
    Xf_trn = X_all.iloc[trn_idx]
    Xf_val = X_all.iloc[val_idx]
    yf_trn = y_all.iloc[trn_idx]
    yf_val = y_all.iloc[val_idx]

    spw_f = (yf_trn == 0).sum() / (yf_trn == 1).sum()

    sc_f = StandardScaler()
    Xf_trn_sc = pd.DataFrame(sc_f.fit_transform(Xf_trn.fillna(0)),
                              columns=Xf_trn.columns, index=Xf_trn.index)
    Xf_val_sc = pd.DataFrame(sc_f.transform(Xf_val.fillna(0)),
                              columns=Xf_val.columns, index=Xf_val.index)

    # 0: LR
    lr_f = LogisticRegression(solver='saga', max_iter=300, random_state=42,
                              class_weight='balanced', n_jobs=-1)
    lr_f.fit(Xf_trn_sc, yf_trn)
    oof_preds[val_idx, 0] = lr_f.predict_proba(Xf_val_sc)[:, 1]
    print(f'  LR done')

    # 1: XGBoost (multi-seed average)
    xgb_acc = np.zeros(len(val_idx))
    for s in SEEDS:
        m = xgb.XGBClassifier(n_estimators=XGB_ESTIMATORS, max_depth=8, learning_rate=XGB_LR,
                               subsample=0.8, colsample_bytree=0.7, reg_alpha=0.1,
                               reg_lambda=1.0, min_child_weight=10, scale_pos_weight=spw_f,
                               max_delta_step=1, random_state=s,
                               n_jobs=-1, eval_metric='auc', tree_method='hist',
                               device='cuda', early_stopping_rounds=EARLY_STOP_ROUNDS)
        m.fit(Xf_trn, yf_trn, eval_set=[(Xf_val, yf_val)], verbose=0)
        xgb_acc += m.predict_proba(Xf_val)[:, 1]
    oof_preds[val_idx, 1] = xgb_acc / len(SEEDS)
    print(f'  XGB ({len(SEEDS)} seeds) done')

    # 2: HGB
    hgb_f = HistGradientBoostingClassifier(
        learning_rate=0.04, max_iter=1200, max_depth=10, max_leaf_nodes=255,
        min_samples_leaf=15, l2_regularization=0.05, class_weight='balanced',
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=50, random_state=42)
    hgb_f.fit(Xf_trn, yf_trn)
    oof_preds[val_idx, 2] = hgb_f.predict_proba(Xf_val)[:, 1]
    print(f'  HGB done')

    # 3: LightGBM (multi-seed average)
    lgb_acc = np.zeros(len(val_idx))
    for s in SEEDS:
        m = fit_lgbm_classifier(
            Xf_trn, yf_trn, Xf_val, yf_val,
            seed=s, n_estimators=LGB_ESTIMATORS, learning_rate=LGB_LR,
            early_stopping_rounds=EARLY_STOP_ROUNDS, log_eval=0,
        )
        lgb_acc += m.predict_proba(Xf_val)[:, 1]
    oof_preds[val_idx, 3] = lgb_acc / len(SEEDS)
    print(f'  LGB ({len(SEEDS)} seeds) done')

    # 4: MLP
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    nf = Xf_trn_sc.shape[1]
    mlp_f = WildlifeMLP(nf).to(dev)
    pw = torch.tensor([(yf_trn==0).sum() / (yf_trn==1).sum()], dtype=torch.float32).to(dev)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt_m = torch.optim.AdamW(mlp_f.parameters(), lr=5e-4, weight_decay=5e-4)
    Xt = torch.tensor(Xf_trn_sc.values, dtype=torch.float32)
    yt = torch.tensor(yf_trn.values, dtype=torch.float32).unsqueeze(1)
    Xvt = torch.tensor(Xf_val_sc.values, dtype=torch.float32)
    dl = DataLoader(TensorDataset(Xt, yt), batch_size=1024, shuffle=True)
    best_auc_f, wait_f, best_st = 0, 0, None
    for ep in range(STACK_MLP_MAX_EPOCHS):
        mlp_f.train()
        for xb, yb in dl:
            xb, yb = xb.to(dev), yb.to(dev)
            opt_m.zero_grad(); crit(mlp_f(xb), yb).backward(); opt_m.step()
        mlp_f.eval()
        with torch.no_grad():
            vp = torch.sigmoid(mlp_f(Xvt.to(dev)).cpu()).numpy().ravel()
            af = roc_auc_score(yf_val, vp)
        if af > best_auc_f:
            best_auc_f = af; best_st = {k: v.cpu().clone() for k, v in mlp_f.state_dict().items()}; wait_f = 0
        else:
            wait_f += 1
        if wait_f >= STACK_MLP_PATIENCE: break
    mlp_f.load_state_dict(best_st); mlp_f.eval()
    with torch.no_grad():
        oof_preds[val_idx, 4] = torch.sigmoid(mlp_f(Xvt.to(dev)).cpu()).numpy().ravel()
    print(f'  MLP done (ep={ep+1})')

    # 5: XGB tuned (multi-seed, uses best params from RandomizedSearchCV)
    xgbt_acc = np.zeros(len(val_idx))
    bp = dict(rs.best_params_)
    bp.pop('n_estimators', None)
    for s in SEEDS:
        m = xgb.XGBClassifier(**bp, n_estimators=XGB_ESTIMATORS, random_state=s, n_jobs=-1,
                               eval_metric='auc', tree_method='hist', device='cuda',
                               early_stopping_rounds=EARLY_STOP_ROUNDS)
        m.fit(Xf_trn, yf_trn, eval_set=[(Xf_val, yf_val)], verbose=0)
        xgbt_acc += m.predict_proba(Xf_val)[:, 1]
    oof_preds[val_idx, 5] = xgbt_acc / len(SEEDS)
    print(f'  XGB_tuned ({len(SEEDS)} seeds) done')

    # 6: CatBoost (budgeted seeds; GPU first with CPU fallback)
    cat_acc = np.zeros(len(val_idx))
    for s in CAT_SEEDS:
        m = fit_catboost(Xf_trn, yf_trn, Xf_val, yf_val, seed=s, iterations=CAT_ITERATIONS)
        cat_acc += m.predict_proba(Xf_val.fillna(0))[:, 1]
    oof_preds[val_idx, 6] = cat_acc / len(CAT_SEEDS)
    print(f'  CatBoost ({len(CAT_SEEDS)} seeds) done')

    # 7: XGBoost DART (single seed; intentionally diverse but slower)
    dart_f = xgb.XGBClassifier(
        n_estimators=DART_ESTIMATORS, max_depth=7, learning_rate=0.012,
        subsample=0.85, colsample_bytree=0.75, min_child_weight=8,
        scale_pos_weight=spw_f, max_delta_step=1, reg_alpha=0.05,
        reg_lambda=1.5, gamma=0.05, booster='dart',
        sample_type='uniform', normalize_type='tree', rate_drop=0.08,
        skip_drop=0.55, random_state=fold_i + 42, n_jobs=-1, eval_metric='auc',
        tree_method='hist', device='cuda', early_stopping_rounds=EARLY_STOP_ROUNDS)
    dart_f.fit(Xf_trn, yf_trn, eval_set=[(Xf_val, yf_val)], verbose=0)
    oof_preds[val_idx, 7] = dart_f.predict_proba(Xf_val)[:, 1]
    print(f'  XGB_DART done')


    # 8: ExtraTrees (balanced bagging signal for stack diversity)

    et_f = ExtraTreesClassifier(

        n_estimators=EXTRA_TREES_N, max_features='sqrt', min_samples_leaf=2,

        class_weight='balanced_subsample', n_jobs=-1, random_state=fold_i + 42

    )

    et_f.fit(Xf_trn.fillna(0), yf_trn)

    oof_preds[val_idx, 8] = et_f.predict_proba(Xf_val.fillna(0))[:, 1]

    print(f'  ExtraTrees done')

    # 9: LightGBM alternate shape for stack diversity
    lgb_alt_acc = np.zeros(len(val_idx))
    for s in SEEDS:
        m = fit_lgbm_classifier(
            Xf_trn, yf_trn, Xf_val, yf_val,
            seed=s + 1000, n_estimators=LGB_ALT_ESTIMATORS, learning_rate=LGB_ALT_LR,
            early_stopping_rounds=EARLY_STOP_ROUNDS, log_eval=0,
            param_overrides=dict(
                num_leaves=511, subsample=0.9, colsample_bytree=0.9,
                min_child_samples=8, reg_alpha=0.0, reg_lambda=0.5,
            ),
        )
        lgb_alt_acc += m.predict_proba(Xf_val)[:, 1]
    oof_preds[val_idx, 9] = lgb_alt_acc / len(SEEDS)
    print(f'  LGB_alt ({len(SEEDS)} seeds) done')

    # 10: CatBoost on retained raw categorical columns
    cat_raw_acc = np.zeros(len(val_idx))
    Xf_trn_raw = X_base.iloc[trn_idx]
    Xf_val_raw = X_base.iloc[val_idx]
    for s in CATBOOST_RAW_SEEDS:
        m = fit_catboost_raw(Xf_trn_raw, yf_trn, Xf_val_raw, yf_val, seed=s, iterations=CAT_RAW_ITERATIONS)
        cat_raw_acc += predict_catboost_raw(m, Xf_val_raw)
    oof_preds[val_idx, 10] = cat_raw_acc / len(CATBOOST_RAW_SEEDS)
    print(f'  CatBoost_raw ({len(CATBOOST_RAW_SEEDS)} seeds) done')

    print(f'  Fold {fold_i+1} elapsed: {(time.time()-t0)/60:.1f} min')

# --- OOF per-model evaluation ---
print(f'\n{"="*60}')
print(f'{"MODEL":25s} | {"OOF Bal.Acc":>12s} | {"OOF AUC":>10s}')
print(f'{"-"*60}')
for i, name in enumerate(stack_model_names):
    t_i, ba_i = find_best_threshold(y_all, oof_preds[:, i])
    auc_i = roc_auc_score(y_all, oof_preds[:, i])
    print(f'{name:25s} | {ba_i:>12.4f} | {auc_i:>10.4f}')

# --- Stacking meta-learner ---
meta_lr = LogisticRegression(solver='saga', max_iter=500, class_weight='balanced',
                             C=1.0, random_state=42)
meta_lr.fit(oof_preds, y_all)
oof_meta_probs = meta_lr.predict_proba(oof_preds)[:, 1]

best_ens_t, best_ens_ba = find_best_threshold(y_all, oof_meta_probs, step=0.001)
oof_meta_auc = roc_auc_score(y_all, oof_meta_probs)

print(f'\n{"STACKED ENSEMBLE":25s} | {best_ens_ba:>12.4f} | {oof_meta_auc:>10.4f}  <<<')
print(f'{"="*60}')
print(f'Optimal threshold: {best_ens_t:.3f}')
print(classification_report(y_all, (oof_meta_probs >= best_ens_t).astype(int)))

# --- Post-stack model selection (no extra model training) ---
def evaluate_oof_candidate(name, probs, step=0.001):
    t, ba = find_best_threshold(y_all, probs, step=step)
    auc = roc_auc_score(y_all, probs)
    return {"name": name, "threshold": t, "balanced_accuracy": ba, "auc": auc, "probs": probs}


def rank_average_columns(pred_matrix):
    return pd.DataFrame(pred_matrix).rank(pct=True).mean(axis=1).to_numpy()


candidate_specs = [
    ("blend_core", ["XGB", "HGB", "LGB", "LGB_alt", "XGB_tuned", "CatBoost", "CatBoost_raw"]),
    ("blend_gpu_core", ["XGB", "LGB", "LGB_alt", "XGB_tuned", "CatBoost", "CatBoost_raw"]),
    ("blend_lgb_cat_xgbt", ["LGB", "LGB_alt", "CatBoost", "CatBoost_raw", "XGB_tuned"]),
    ("blend_lgb_cat", ["LGB", "LGB_alt", "CatBoost", "CatBoost_raw"]),
    ("blend_cat_family", ["CatBoost", "CatBoost_raw"]),
    ("blend_lgb_family", ["LGB", "LGB_alt"]),
    ("blend_no_mlp", ["XGB", "HGB", "LGB", "LGB_alt", "XGB_tuned", "CatBoost", "CatBoost_raw", "XGB_DART", "ExtraTrees"]),
    ("blend_tree_all", ["XGB", "HGB", "LGB", "LGB_alt", "XGB_tuned", "CatBoost", "CatBoost_raw", "XGB_DART", "ExtraTrees"]),
    ("blend_no_cpu_bagging", ["XGB", "LGB", "LGB_alt", "MLP", "XGB_tuned", "CatBoost", "CatBoost_raw"]),
]

rank_candidate_specs = [
    ("rank_gpu_core", ["XGB", "LGB", "LGB_alt", "XGB_tuned", "CatBoost", "CatBoost_raw"]),
    ("rank_tree_all", ["XGB", "HGB", "LGB", "LGB_alt", "XGB_tuned", "CatBoost", "CatBoost_raw", "XGB_DART", "ExtraTrees"]),
    ("rank_lgb_cat_xgbt", ["LGB", "LGB_alt", "CatBoost", "CatBoost_raw", "XGB_tuned"]),
]

weighted_candidate_specs = [
    ("weighted_gpu_core", ["XGB", "LGB", "LGB_alt", "XGB_tuned", "CatBoost", "CatBoost_raw"]),
    ("weighted_tree_all", ["XGB", "HGB", "LGB", "LGB_alt", "XGB_tuned", "CatBoost", "CatBoost_raw", "XGB_DART", "ExtraTrees"]),
    ("weighted_all", stack_model_names),
]


def optimize_weighted_blend(name, names, n_trials=96, seed=42):
    idxs = [stack_model_names.index(n) for n in names if n in stack_model_names]
    mat = oof_preds[:, idxs]
    rng = np.random.RandomState(seed)

    weight_candidates = [np.ones(len(idxs)) / len(idxs)]
    for j in range(len(idxs)):
        w = np.zeros(len(idxs)); w[j] = 1.0
        weight_candidates.append(w)
    for alpha in [0.45, 0.8, 1.5, 3.0]:
        draws = max(1, n_trials // 4)
        weight_candidates.extend(rng.dirichlet(np.full(len(idxs), alpha), size=draws))

    best = None
    best_weights = None
    for weights in weight_candidates:
        probs = mat @ weights
        cand = evaluate_oof_candidate(name, probs, step=0.002)
        if best is None or cand["balanced_accuracy"] > best["balanced_accuracy"]:
            best = cand
            best_weights = weights

    weighted_blend_weights[name] = {
        "names": [stack_model_names[i] for i in idxs],
        "weights": best_weights,
    }
    weights_text = ", ".join(
        f"{n}={w:.3f}" for n, w in zip(weighted_blend_weights[name]["names"], best_weights)
    )
    print(f"Weighted candidate {name}: {weights_text}")
    return best


oof_candidates = [evaluate_oof_candidate("stack_full", oof_meta_probs)]
for cand_name, names in candidate_specs:
    idxs = [stack_model_names.index(n) for n in names if n in stack_model_names]
    if idxs:
        probs = oof_preds[:, idxs].mean(axis=1)
        oof_candidates.append(evaluate_oof_candidate(cand_name, probs))

for cand_name, names in rank_candidate_specs:
    idxs = [stack_model_names.index(n) for n in names if n in stack_model_names]
    if idxs:
        probs = rank_average_columns(oof_preds[:, idxs])
        oof_candidates.append(evaluate_oof_candidate(cand_name, probs))

weighted_blend_weights = {}
for cand_name, names in weighted_candidate_specs:
    oof_candidates.append(optimize_weighted_blend(cand_name, names, seed=42 + len(oof_candidates)))

oof_candidate_df = pd.DataFrame([
    {k: v for k, v in c.items() if k != "probs"} for c in oof_candidates
]).sort_values("balanced_accuracy", ascending=False)
print("\nOOF candidate comparison:")
print(oof_candidate_df.to_string(index=False))

best_submission_candidate = max(oof_candidates, key=lambda c: c["balanced_accuracy"])
print(
    f"\nSelected submission candidate: {best_submission_candidate['name']} "
    f"threshold={best_submission_candidate['threshold']:.3f} "
    f"OOF BA={best_submission_candidate['balanced_accuracy']:.4f}"
)

# --- OOF overfit audit: threshold/model selection stability check ---
audit_idx = np.arange(N)
audit_fit_idx, audit_check_idx = train_test_split(
    audit_idx, test_size=0.25, random_state=2026, stratify=y_all
)

def audit_candidate_generalization(c):
    probs = c["probs"]
    t_fit, fit_ba = find_best_threshold(y_all.iloc[audit_fit_idx], probs[audit_fit_idx], step=0.001)
    check_preds = (probs[audit_check_idx] >= t_fit).astype(int)
    check_ba = balanced_accuracy_score(y_all.iloc[audit_check_idx], check_preds)
    return {
        "name": c["name"],
        "full_oof_ba": c["balanced_accuracy"],
        "audit_fit_ba": fit_ba,
        "audit_check_ba": check_ba,
        "audit_gap": fit_ba - check_ba,
        "threshold_fit_75pct": t_fit,
    }

audit_rows = [audit_candidate_generalization(c) for c in oof_candidates]
audit_df = pd.DataFrame(audit_rows).sort_values("audit_check_ba", ascending=False)
print("\nOOF overfit audit (threshold fit on 75%, evaluated on held-out 25% of OOF rows):")
print(audit_df.to_string(index=False))

meta_audit = LogisticRegression(solver='saga', max_iter=500, class_weight='balanced', C=1.0, random_state=2026)
meta_audit.fit(oof_preds[audit_fit_idx], y_all.iloc[audit_fit_idx])
meta_fit_probs = meta_audit.predict_proba(oof_preds[audit_fit_idx])[:, 1]
meta_check_probs = meta_audit.predict_proba(oof_preds[audit_check_idx])[:, 1]
meta_audit_t, meta_audit_fit_ba = find_best_threshold(y_all.iloc[audit_fit_idx], meta_fit_probs, step=0.001)
meta_audit_check_ba = balanced_accuracy_score(
    y_all.iloc[audit_check_idx], (meta_check_probs >= meta_audit_t).astype(int)
)
meta_audit_gap = meta_audit_fit_ba - meta_audit_check_ba
print(
    f"\nMeta-learner audit: fit BA={meta_audit_fit_ba:.4f}, "
    f"held-out OOF BA={meta_audit_check_ba:.4f}, gap={meta_audit_gap:.4f}, "
    f"threshold={meta_audit_t:.3f}"
)

selected_audit = audit_df.loc[audit_df["name"].eq(best_submission_candidate["name"])].iloc[0]
selected_gap = float(selected_audit["audit_gap"])
if selected_gap <= 0.010:
    selected_overfit_risk = "low"
elif selected_gap <= 0.025:
    selected_overfit_risk = "moderate"
else:
    selected_overfit_risk = "high"
print(
    f"Selected candidate OOF overfit risk: {selected_overfit_risk} "
    f"(75/25 audit gap={selected_gap:.4f})"
)

print(f'Total stacking time: {(time.time()-t0)/60:.1f} min')





## Phase 5: Generate Competition Submission

Retrain every base model on **100% of training data** (no hold-out), including the alternate LightGBM and raw-categorical CatBoost variants, generate test
predictions, then use the best OOF-selected candidate from the stacking cell. The cell writes
`submission.csv` plus nearby-threshold variants for carefully limited Kaggle threshold checks.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Retrain ALL base models on 100% training data, then predict test
#  via the fitted stacking meta-learner + optimized threshold.
# ═══════════════════════════════════════════════════════════════════
print('Retraining all models on 100% training data for final submission...\n')
t0_sub = time.time()

# Scaled full training set + test set
sc_full = StandardScaler()
X_all_sc = pd.DataFrame(sc_full.fit_transform(X_all.fillna(0)),
                         columns=X_all.columns, index=X_all.index)
X_sub_sc_full = pd.DataFrame(sc_full.transform(X_sub.fillna(0)),
                              columns=X_sub.columns, index=X_sub.index)

test_base_preds = np.zeros((len(X_sub), len(stack_model_names)), dtype=np.float64)
neg_all = (y_all == 0).sum(); pos_all = (y_all == 1).sum()

# 0: LR
lr_full = LogisticRegression(solver='saga', max_iter=300, random_state=42,
                              class_weight='balanced', n_jobs=-1)
lr_full.fit(X_all_sc, y_all)
test_base_preds[:, 0] = lr_full.predict_proba(X_sub_sc_full)[:, 1]
print('  LR retrained')

# 1: XGB (multi-seed)
for s in SEEDS:
    m = xgb.XGBClassifier(n_estimators=XGB_ESTIMATORS, max_depth=8, learning_rate=XGB_LR,
                           subsample=0.8, colsample_bytree=0.7, reg_alpha=0.1,
                           reg_lambda=1.0, min_child_weight=10,
                           scale_pos_weight=neg_all / pos_all, max_delta_step=1, random_state=s,
                           n_jobs=-1, eval_metric='auc', tree_method='hist', device='cuda')
    m.fit(X_all, y_all, verbose=0)
    test_base_preds[:, 1] += m.predict_proba(X_sub)[:, 1]
test_base_preds[:, 1] /= len(SEEDS)
print(f'  XGB retrained ({len(SEEDS)} seeds)')

# 2: HGB
hgb_full = HistGradientBoostingClassifier(
    learning_rate=0.04, max_iter=1200, max_depth=10, max_leaf_nodes=255,
    min_samples_leaf=15, l2_regularization=0.05, class_weight='balanced', random_state=42)
hgb_full.fit(X_all, y_all)
test_base_preds[:, 2] = hgb_full.predict_proba(X_sub)[:, 1]
print('  HGB retrained')

# 3: LGB (multi-seed)
for s in SEEDS:
    m = fit_lgbm_classifier(
        X_all, y_all,
        seed=s, n_estimators=LGB_ESTIMATORS, learning_rate=LGB_LR,
        early_stopping_rounds=None, log_eval=0,
    )
    test_base_preds[:, 3] += m.predict_proba(X_sub)[:, 1]
test_base_preds[:, 3] /= len(SEEDS)
print(f'  LGB retrained ({len(SEEDS)} seeds)')

# 4: MLP
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mlp_full = WildlifeMLP(X_all_sc.shape[1]).to(dev)
pw = torch.tensor([neg_all / pos_all], dtype=torch.float32).to(dev)
crit = nn.BCEWithLogitsLoss(pos_weight=pw)
opt_m = torch.optim.AdamW(mlp_full.parameters(), lr=5e-4, weight_decay=5e-4)
Xt = torch.tensor(X_all_sc.values, dtype=torch.float32)
yt = torch.tensor(y_all.values, dtype=torch.float32).unsqueeze(1)
dl = DataLoader(TensorDataset(Xt, yt), batch_size=1024, shuffle=True)
for ep in range(FINAL_MLP_EPOCHS):
    mlp_full.train()
    for xb, yb in dl:
        xb, yb = xb.to(dev), yb.to(dev)
        opt_m.zero_grad(); crit(mlp_full(xb), yb).backward(); opt_m.step()
mlp_full.eval()
Xst = torch.tensor(X_sub_sc_full.values, dtype=torch.float32)
with torch.no_grad():
    test_base_preds[:, 4] = torch.sigmoid(mlp_full(Xst.to(dev)).cpu()).numpy().ravel()
print(f'  MLP retrained ({FINAL_MLP_EPOCHS} epochs)')

# 5: XGB tuned (multi-seed)
bp = dict(rs.best_params_)
bp.pop('n_estimators', None)
for s in SEEDS:
    m = xgb.XGBClassifier(**bp, n_estimators=XGB_ESTIMATORS, random_state=s, n_jobs=-1,
                           eval_metric='auc', tree_method='hist', device='cuda')
    m.fit(X_all, y_all, verbose=0)
    test_base_preds[:, 5] += m.predict_proba(X_sub)[:, 1]
test_base_preds[:, 5] /= len(SEEDS)
print(f'  XGB_tuned retrained ({len(SEEDS)} seeds)')

# 6: CatBoost (budgeted seeds)
for s in CAT_SEEDS:
    m = fit_catboost(X_all, y_all, seed=s, iterations=CAT_ITERATIONS)
    test_base_preds[:, 6] += m.predict_proba(X_sub.fillna(0))[:, 1]
test_base_preds[:, 6] /= len(CAT_SEEDS)
print(f'  CatBoost retrained ({len(CAT_SEEDS)} seeds)')

# 7: XGB DART
m = xgb.XGBClassifier(
    n_estimators=DART_ESTIMATORS, max_depth=7, learning_rate=0.012,
    subsample=0.85, colsample_bytree=0.75, min_child_weight=8,
    scale_pos_weight=neg_all / pos_all, max_delta_step=1, reg_alpha=0.05,
    reg_lambda=1.5, gamma=0.05, booster='dart',
    sample_type='uniform', normalize_type='tree', rate_drop=0.08,
    skip_drop=0.55, random_state=42, n_jobs=-1, eval_metric='auc',
    tree_method='hist', device='cuda')
m.fit(X_all, y_all, verbose=0)
test_base_preds[:, 7] = m.predict_proba(X_sub)[:, 1]
print('  XGB_DART retrained')


et_full = ExtraTreesClassifier(

    n_estimators=EXTRA_TREES_N, max_features='sqrt', min_samples_leaf=2,

    class_weight='balanced_subsample', n_jobs=-1, random_state=42

)

et_full.fit(X_all.fillna(0), y_all)

test_base_preds[:, 8] = et_full.predict_proba(X_sub.fillna(0))[:, 1]

print('  ExtraTrees retrained')

# 9: LGB alternate
for s in SEEDS:
    m = fit_lgbm_classifier(
        X_all, y_all,
        seed=s + 1000, n_estimators=LGB_ALT_ESTIMATORS, learning_rate=LGB_ALT_LR,
        early_stopping_rounds=None, log_eval=0,
        param_overrides=dict(
            num_leaves=511, subsample=0.9, colsample_bytree=0.9,
            min_child_samples=8, reg_alpha=0.0, reg_lambda=0.5,
        ),
    )
    test_base_preds[:, 9] += m.predict_proba(X_sub)[:, 1]
test_base_preds[:, 9] /= len(SEEDS)
print(f'  LGB_alt retrained ({len(SEEDS)} seeds)')

# 10: CatBoost raw categorical
for s in CATBOOST_RAW_SEEDS:
    m = fit_catboost_raw(X_base, y_all, seed=s, iterations=CAT_RAW_ITERATIONS)
    test_base_preds[:, 10] += predict_catboost_raw(m, X_sub_base)
test_base_preds[:, 10] /= len(CATBOOST_RAW_SEEDS)
print(f'  CatBoost_raw retrained ({len(CATBOOST_RAW_SEEDS)} seeds)')

# --- Apply selected OOF candidate + threshold ---
def rank_average_test_columns(pred_matrix):
    return pd.DataFrame(pred_matrix).rank(pct=True).mean(axis=1).to_numpy()


sub_meta_probs = meta_lr.predict_proba(test_base_preds)[:, 1]
test_candidate_probs = {"stack_full": sub_meta_probs}
for cand_name, names in candidate_specs:
    idxs = [stack_model_names.index(n) for n in names if n in stack_model_names]
    if idxs:
        test_candidate_probs[cand_name] = test_base_preds[:, idxs].mean(axis=1)

if "rank_candidate_specs" in globals():
    for cand_name, names in rank_candidate_specs:
        idxs = [stack_model_names.index(n) for n in names if n in stack_model_names]
        if idxs:
            test_candidate_probs[cand_name] = rank_average_test_columns(test_base_preds[:, idxs])

if "weighted_blend_weights" in globals():
    for cand_name, spec in weighted_blend_weights.items():
        idxs = [stack_model_names.index(n) for n in spec["names"] if n in stack_model_names]
        weights = np.asarray(spec["weights"], dtype=float)
        if idxs and len(idxs) == len(weights):
            test_candidate_probs[cand_name] = test_base_preds[:, idxs] @ weights

candidate_thresholds = (
    {c["name"]: c["threshold"] for c in oof_candidates}
    if "oof_candidates" in globals() else {"stack_full": best_ens_t}
)
selected_name = best_submission_candidate["name"] if "best_submission_candidate" in globals() else "stack_full"
selected_threshold = candidate_thresholds.get(selected_name, best_ens_t)
selected_probs = test_candidate_probs.get(selected_name, sub_meta_probs)
sub_preds = (selected_probs >= selected_threshold).astype(int)

submission = pd.DataFrame({
    'INDEX_NR': submission_ids,
    'INDICATED_DAMAGE': sub_preds
})
submission.to_csv('submission.csv', index=False)

# These are all model-derived threshold/candidate variants. Submit sparingly,
# using the public leaderboard direction to choose the next threshold.
variant_deltas = [-0.05, -0.04, -0.03, -0.02, -0.015, -0.01, -0.005,
                  0.005, 0.01, 0.015, 0.02, 0.03, 0.04, 0.05]
if "oof_candidate_df" in globals():
    variant_candidate_names = [n for n in oof_candidate_df.head(3)["name"] if n in test_candidate_probs]
else:
    variant_candidate_names = [selected_name]

for cand_name in variant_candidate_names:
    probs = test_candidate_probs[cand_name]
    base_t = candidate_thresholds.get(cand_name, selected_threshold)
    for delta in variant_deltas:
        t = min(max(base_t + delta, 0.001), 0.999)
        variant_preds = (probs >= t).astype(int)
        variant = pd.DataFrame({
            'INDEX_NR': submission_ids,
            'INDICATED_DAMAGE': variant_preds,
        })
        variant_name = f"submission_{cand_name}_thr_{t:.3f}.csv".replace('.', 'p')
        variant.to_csv(variant_name, index=False)
        print(f"Variant saved: {variant_name}  threshold={t:.3f}  damage_rate={variant_preds.mean():.4f}")

print(f'\nsubmission.csv saved  ({len(submission):,} rows)')
print(f'Selected candidate: {selected_name}')
print(f'Selected threshold: {selected_threshold:.3f}')
print(f'Predicted damage rate: {sub_preds.mean():.4f}')
print(f'Distribution: 0={int((sub_preds==0).sum()):,}  1={int((sub_preds==1).sum()):,}')
print(f'Retrain time: {(time.time()-t0_sub)/60:.1f} min')
print(f'\nFirst 10 rows:\n{submission.head(10).to_string(index=False)}')

try:
    colab_files.download('submission.csv')
except Exception:
    print('\n(Auto-download skipped — file saved to working directory.)')



